<a href="https://colab.research.google.com/github/fallouseck1-beep/Fallou-Seck/blob/main/terrafisc_v4_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ TERRAFISC v4 — Système de Gestion Administrative
**Logo TF · Charte Blanc / Marron / Or · Emails auto · Photos profil · Gestion statuts**

**Exécuter dans l'ordre : Cellule 1 → 2 → 3 → 4 → 5**

In [ ]:
!pip install pyngrok --quiet

## 🗄️ Cellule 2 — Base de données & initialisation

In [ ]:
import sqlite3, hashlib, unicodedata, re, random, string
from datetime import date

DB = 'terrafisc.db'

def get_db():
    c = sqlite3.connect(DB)
    c.row_factory = sqlite3.Row
    c.execute('PRAGMA foreign_keys=ON')
    return c

def h(p): return hashlib.sha256(p.encode()).hexdigest()

def clean_str(s):
    if not s: return ''
    s = s.lower().strip()
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode()
    s = re.sub(r'[^a-z]', '', s)
    return s

def generate_email(nom, prenom, conn):
    """Génère un email unique à partir des noms/prénoms avec caractère spécial et chiffre."""
    base = clean_str(prenom) + '.' + clean_str(nom)
    if not base.replace('.',''):
        base = 'personnel'
    rows = conn.execute("SELECT email FROM personnel WHERE email LIKE ?", (base + '.%@terrafisc.sn',)).fetchall()
    used = set()
    for row in rows:
        m = re.search(r'\.(\d+)@terrafisc\.sn$', row[0])
        if m:
            used.add(int(m.group(1)))
    n = 1
    while n in used:
        n += 1
    return f"{base}.{n}@terrafisc.sn"

def gen_password():
    """Génère un mot de passe sécurisé avec lettres, chiffres et caractère spécial."""
    up = random.choice(string.ascii_uppercase)
    sp = random.choice('@#$!')
    mid = ''.join(random.choices(string.ascii_lowercase + string.digits, k=5))
    end = str(random.randint(10, 99))
    return up + mid + sp + end

def init_db():
    conn = get_db(); c = conn.cursor()
    c.executescript('''
    CREATE TABLE IF NOT EXISTS bureau(
        bureau_id    INTEGER PRIMARY KEY AUTOINCREMENT,
        nom          VARCHAR(80) NOT NULL,
        code         VARCHAR(10),
        description  TEXT
    );
    CREATE TABLE IF NOT EXISTS personnel(
        personnel_id    INTEGER PRIMARY KEY AUTOINCREMENT,
        nom             VARCHAR(30) NOT NULL,
        prenom          VARCHAR(30),
        email           TEXT UNIQUE NOT NULL,
        mot_de_passe    TEXT NOT NULL,
        telephone       VARCHAR(15),
        dateNaissance   DATE,
        fonction        TEXT,
        role            TEXT NOT NULL CHECK(role IN ("chef_centre","chef_bureau","controleur","agent")),
        adresse         VARCHAR(100),
        superieur_id    INTEGER REFERENCES personnel(personnel_id),
        bureau_id       INTEGER REFERENCES bureau(bureau_id),
        annee_integration INTEGER,
        actif           INTEGER DEFAULT 1,
        statut_emploi   TEXT DEFAULT "Actif",
        photo           TEXT
    );
    CREATE TABLE IF NOT EXISTS activite(
        activite_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        nom           VARCHAR(100),
        type_activite VARCHAR(30),
        description   TEXT,
        dateDebut     DATE,
        dateFin       DATE,
        statut        VARCHAR(20) DEFAULT "Planifiee",
        bureau_id     INTEGER REFERENCES bureau(bureau_id),
        propose_par   INTEGER REFERENCES personnel(personnel_id),
        validee_par   INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS proposition_activite(
        prop_id       INTEGER PRIMARY KEY AUTOINCREMENT,
        description   TEXT,
        type_activite VARCHAR(30),
        date_prop     DATE,
        statut        VARCHAR(20) DEFAULT "En attente",
        propose_par   INTEGER REFERENCES personnel(personnel_id),
        destine_a     INTEGER REFERENCES personnel(personnel_id),
        commentaire_reponse TEXT
    );
    CREATE TABLE IF NOT EXISTS performance(
        performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
        efficacite     INTEGER CHECK(efficacite BETWEEN 0 AND 100),
        note           INTEGER CHECK(note BETWEEN 0 AND 20),
        prime          VARCHAR(50),
        commentaire    TEXT,
        personnel_id   INTEGER REFERENCES personnel(personnel_id),
        evalue_par     INTEGER REFERENCES personnel(personnel_id),
        date_eval      DATE,
        mois           INTEGER,
        annee          INTEGER
    );
    CREATE TABLE IF NOT EXISTS employe_mois(
        em_id          INTEGER PRIMARY KEY AUTOINCREMENT,
        personnel_id   INTEGER REFERENCES personnel(personnel_id),
        mois           INTEGER NOT NULL,
        annee          INTEGER NOT NULL,
        note_finale    REAL,
        motif          TEXT,
        designe_par    INTEGER REFERENCES personnel(personnel_id),
        date_designation DATE,
        UNIQUE(mois, annee)
    );
    CREATE TABLE IF NOT EXISTS tache(
        tache_id     INTEGER PRIMARY KEY AUTOINCREMENT,
        libelle      VARCHAR(100),
        description  TEXT,
        dateDebut    DATE,
        dateFin      DATE,
        statut       VARCHAR(30) DEFAULT "Non demarre",
        priorite     VARCHAR(20) DEFAULT "Normale",
        activite_id  INTEGER REFERENCES activite(activite_id),
        performance_id INTEGER REFERENCES performance(performance_id)
    );
    CREATE TABLE IF NOT EXISTS affecter(
        affect_id        INTEGER PRIMARY KEY AUTOINCREMENT,
        tache_id         INTEGER REFERENCES tache(tache_id),
        personnel_id     INTEGER REFERENCES personnel(personnel_id),
        role_affect      TEXT DEFAULT "Executant",
        date_affectation DATE,
        date_retrait     DATE,
        actif            INTEGER DEFAULT 1,
        assigne_par      INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS notification(
        notification_id INTEGER PRIMARY KEY AUTOINCREMENT,
        titre           TEXT NOT NULL,
        description     TEXT NOT NULL,
        type_notif      VARCHAR(30) DEFAULT "info",
        dateEnvoie      DATETIME DEFAULT (CURRENT_TIMESTAMP),
        statut          VARCHAR(20) DEFAULT "Non lue",
        tache_id        INTEGER REFERENCES tache(tache_id),
        destinataire_id INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS signalement(
        signalement_id INTEGER PRIMARY KEY AUTOINCREMENT,
        description    TEXT,
        dateEnvoie     DATE,
        statut         VARCHAR(30) DEFAULT "Ouvert",
        reponse        TEXT,
        personnel_id   INTEGER REFERENCES personnel(personnel_id),
        tache_id       INTEGER REFERENCES tache(tache_id),
        destine_a      INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS idee(
        idee_id     INTEGER PRIMARY KEY AUTOINCREMENT,
        description TEXT,
        dateRecue   DATE,
        statut      VARCHAR(30) DEFAULT "Soumise",
        activite_id INTEGER REFERENCES activite(activite_id)
    );
    CREATE TABLE IF NOT EXISTS avis(
        avis_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        commentaire  TEXT,
        note         INTEGER CHECK(note BETWEEN 1 AND 5),
        dateEnvoie   DATE,
        personnel_id INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS compteRendu(
        cr_id        INTEGER PRIMARY KEY AUTOINCREMENT,
        dateRendue   DATE,
        contenu      TEXT NOT NULL,
        statut       VARCHAR(20) DEFAULT "Brouillon",
        personnel_id INTEGER REFERENCES personnel(personnel_id),
        tache_id     INTEGER REFERENCES tache(tache_id)
    );
    CREATE TABLE IF NOT EXISTS telepaiement(
        tp_id            INTEGER PRIMARY KEY AUTOINCREMENT,
        reference        TEXT UNIQUE NOT NULL,
        contribuable_nom TEXT NOT NULL,
        contribuable_nif TEXT,
        type_impot       TEXT NOT NULL,
        montant          REAL NOT NULL,
        montant_paye     REAL DEFAULT 0,
        statut           VARCHAR(30) DEFAULT "En attente",
        mode_paiement    TEXT,
        date_echeance    DATE,
        date_paiement    DATE,
        bureau_id        INTEGER REFERENCES bureau(bureau_id),
        agent_id         INTEGER REFERENCES personnel(personnel_id),
        notes            TEXT
    );
    ''')
    conn.commit()
    # Créer un compte admin par défaut s'il n'existe pas
    if c.execute("SELECT COUNT(*) FROM personnel WHERE role='chef_centre'").fetchone()[0] == 0:
        admin_pwd = 'TF@Admin2025'
        c.execute("""INSERT INTO personnel(nom,prenom,email,mot_de_passe,fonction,role,actif,annee_integration)
            VALUES(?,?,?,?,?,?,1,?)""",
            ('ADMIN','Chef','admin@terrafisc.sn',h(admin_pwd),'Directeur du Centre des Impôts','chef_centre',date.today().year))
        conn.commit()
        print('╔' + '═'*52 + '╗')
        print('║  COMPTE CHEF DE CENTRE CRÉÉ AUTOMATIQUEMENT     ║')
        print('╠' + '═'*52 + '╣')
        print('║  Email    :  admin@terrafisc.sn                  ║')
        print('║  Mot de passe :  TF@Admin2025                    ║')
        print('╚' + '═'*52 + '╝')
    else:
        print('ℹ️  Base de données déjà initialisée')
    conn.close()
    print('✅ Base de données TERRAFISC prête')

init_db()


ℹ️  Base de données déjà initialisée
✅ Base de données TERRAFISC prête


## 🚀 Cellule 3 — Backend Flask (API + Logique métier)

In [ ]:
!pip install flask_cors --quiet
from flask import Flask, request, jsonify, render_template_string, session
from flask_cors import CORS
import sqlite3, threading, hashlib, json, re, unicodedata, random, string
from datetime import date, datetime

app = Flask(__name__)
app.secret_key = 'terrafisc_tf_2025_secret_key'
CORS(app, supports_credentials=True)

def h(p): return hashlib.sha256(p.encode()).hexdigest()
def qry(sql, params=(), one=False):
    conn = get_db()
    rows = [dict(r) for r in conn.execute(sql, params).fetchall()]
    conn.close()
    return rows[0] if (one and rows) else (None if (one and not rows) else rows)
def exe(sql, params=()):
    conn = get_db(); cur = conn.execute(sql, params); conn.commit(); lid = cur.lastrowid; conn.close(); return lid
def notif(titre, desc, type_n, dest_id, tache_id=None):
    if dest_id:
        exe('INSERT INTO notification(titre,description,type_notif,tache_id,destinataire_id) VALUES(?,?,?,?,?)',
            (titre, desc, type_n, tache_id, dest_id))
def uid(): return session.get('user_id')
def role(): return session.get('role')
def auth(): return 'user_id' in session
MOIS_FR = ['','Janvier','Février','Mars','Avril','Mai','Juin','Juillet','Août','Septembre','Octobre','Novembre','Décembre']

# ═══════════════════════════════════════════════════════════════
#  AUTH
# ═══════════════════════════════════════════════════════════════
@app.route('/api/login', methods=['POST'])
def login():
    d = request.json
    u = qry('SELECT * FROM personnel WHERE email=? AND mot_de_passe=? AND actif=1', (d['email'], h(d['password'])), one=True)
    if not u: return jsonify({'error': 'Email ou mot de passe incorrect, ou compte désactivé'}), 401
    session['user_id'] = u['personnel_id']
    session['role'] = u['role']
    u.pop('mot_de_passe', None)
    return jsonify(u)

@app.route('/api/logout', methods=['POST'])
def logout():
    session.clear(); return jsonify({'ok': True})

# ═══════════════════════════════════════════════════════════════
#  PROFIL (mon profil)
# ═══════════════════════════════════════════════════════════════
@app.route('/api/me', methods=['GET','PUT'])
def my_profile():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid()
    if request.method == 'PUT':
        d = request.json
        exe('UPDATE personnel SET telephone=?,adresse=?,photo=?,dateNaissance=? WHERE personnel_id=?',
            (d.get('telephone'), d.get('adresse'), d.get('photo'), d.get('dateNaissance'), u))
        return jsonify({'ok': True})
    p = qry('SELECT p.*,b.nom bureau_nom,s.nom sup_nom,s.prenom sup_prenom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel s ON p.superieur_id=s.personnel_id WHERE p.personnel_id=?', (u,), one=True)
    if p: p.pop('mot_de_passe', None)
    return jsonify(p)

# ═══════════════════════════════════════════════════════════════
#  DASHBOARD
# ═══════════════════════════════════════════════════════════════
@app.route('/api/dashboard')
def dashboard():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    r = role(); u = uid()
    nb_notifs = qry("SELECT COUNT(*) c FROM notification WHERE destinataire_id=? AND statut='Non lue'", (u,), one=True)['c']
    if r == 'chef_centre':
        return jsonify({
            'nb_bureaux': qry('SELECT COUNT(*) c FROM bureau', one=True)['c'],
            'nb_personnel': qry('SELECT COUNT(*) c FROM personnel WHERE actif=1', one=True)['c'],
            'nb_activites': qry('SELECT COUNT(*) c FROM activite', one=True)['c'],
            'nb_taches': qry('SELECT COUNT(*) c FROM tache', one=True)['c'],
            'nb_notifs': nb_notifs,
            'taches_statut': qry('SELECT statut, COUNT(*) n FROM tache GROUP BY statut'),
            'activites_statut': qry('SELECT statut, COUNT(*) n FROM activite GROUP BY statut'),
            'bureaux_activites': qry('SELECT b.nom, COUNT(a.activite_id) n FROM bureau b LEFT JOIN activite a ON b.bureau_id=a.bureau_id GROUP BY b.nom'),
            'top_performances': qry('SELECT p.nom,p.prenom,pf.note,pf.efficacite,b.nom bureau_nom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id ORDER BY pf.note DESC LIMIT 8'),
            'employe_mois': qry('SELECT em.*,p.nom,p.prenom,b.nom bureau_nom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id ORDER BY em.annee DESC,em.mois DESC LIMIT 3'),
            'tp_stats': qry("SELECT statut,COUNT(*) n,SUM(montant) total FROM telepaiement GROUP BY statut"),
            'nb_suspendus': qry("SELECT COUNT(*) c FROM personnel WHERE statut_emploi='Mis en pied' AND actif=1", one=True)['c']
        })
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify({
            'nb_controleurs': qry('SELECT COUNT(*) c FROM personnel WHERE superieur_id=? AND role="controleur" AND actif=1', (u,), one=True)['c'],
            'nb_agents': qry('SELECT COUNT(*) c FROM personnel WHERE bureau_id=? AND role="agent" AND actif=1', (bid,), one=True)['c'],
            'nb_activites': qry('SELECT COUNT(*) c FROM activite WHERE bureau_id=?', (bid,), one=True)['c'],
            'nb_taches': qry('SELECT COUNT(*) c FROM tache t JOIN activite a ON t.activite_id=a.activite_id WHERE a.bureau_id=?', (bid,), one=True)['c'],
            'nb_notifs': nb_notifs,
            'taches_statut': qry('SELECT t.statut,COUNT(*) n FROM tache t JOIN activite a ON t.activite_id=a.activite_id WHERE a.bureau_id=? GROUP BY t.statut', (bid,)),
            'propositions_recues': qry("SELECT COUNT(*) c FROM proposition_activite WHERE destine_a=? AND statut='En attente'", (u,), one=True)['c'],
            'signalements_ouverts': qry("SELECT COUNT(*) c FROM signalement WHERE destine_a=? AND statut='Ouvert'", (u,), one=True)['c'],
            'employe_mois': qry('SELECT em.*,p.nom,p.prenom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.bureau_id=? ORDER BY em.annee DESC,em.mois DESC LIMIT 3', (bid,)),
            'tp_bureau': qry('SELECT statut,COUNT(*) n,SUM(montant) total FROM telepaiement WHERE bureau_id=? GROUP BY statut', (bid,))
        })
    elif r == 'controleur':
        return jsonify({
            'nb_agents': qry('SELECT COUNT(*) c FROM personnel WHERE superieur_id=? AND actif=1', (u,), one=True)['c'],
            'nb_taches_assignees': qry('SELECT COUNT(DISTINCT tache_id) c FROM affecter WHERE assigne_par=? AND actif=1', (u,), one=True)['c'],
            'nb_notifs': nb_notifs,
            'taches_statut': qry('SELECT t.statut,COUNT(*) n FROM tache t JOIN affecter af ON t.tache_id=af.tache_id WHERE af.assigne_par=? AND af.actif=1 GROUP BY t.statut', (u,)),
            'signalements_recus': qry("SELECT COUNT(*) c FROM signalement WHERE destine_a=? AND statut='Ouvert'", (u,), one=True)['c'],
            'mes_taches': qry('SELECT t.*,a.nom act_nom FROM tache t JOIN affecter af ON t.tache_id=af.tache_id JOIN activite a ON t.activite_id=a.activite_id WHERE af.personnel_id=? AND af.actif=1 ORDER BY t.dateFin', (u,))
        })
    else:
        return jsonify({
            'nb_taches': qry('SELECT COUNT(*) c FROM affecter WHERE personnel_id=? AND actif=1', (u,), one=True)['c'],
            'nb_terminees': qry("SELECT COUNT(*) c FROM tache t JOIN affecter af ON t.tache_id=af.tache_id WHERE af.personnel_id=? AND af.actif=1 AND t.statut='Terminee'", (u,), one=True)['c'],
            'nb_notifs': nb_notifs,
            'ma_performance': qry('SELECT * FROM performance WHERE personnel_id=? ORDER BY date_eval DESC LIMIT 1', (u,), one=True),
            'mes_taches': qry('SELECT t.*,a.nom act_nom FROM tache t JOIN affecter af ON t.tache_id=af.tache_id JOIN activite a ON t.activite_id=a.activite_id WHERE af.personnel_id=? AND af.actif=1 ORDER BY t.dateFin', (u,)),
            'employe_mois_courant': qry('SELECT em.*,p.nom,p.prenom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id WHERE em.mois=? AND em.annee=?', (date.today().month, date.today().year), one=True)
        })

# ═══════════════════════════════════════════════════════════════
#  BUREAUX
# ═══════════════════════════════════════════════════════════════
@app.route('/api/bureaux', methods=['GET','POST'])
def bureaux():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if request.method == 'POST':
        if role() != 'chef_centre': return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        lid = exe('INSERT INTO bureau(nom,code,description) VALUES(?,?,?)', (d['nom'], d.get('code'), d.get('description')))
        return jsonify({'bureau_id': lid}), 201
    return jsonify(qry('SELECT b.*,(SELECT COUNT(*) FROM personnel WHERE bureau_id=b.bureau_id AND actif=1) nb_pers,(SELECT COUNT(*) FROM activite WHERE bureau_id=b.bureau_id) nb_act FROM bureau b'))

# ═══════════════════════════════════════════════════════════════
#  PERSONNEL
# ═══════════════════════════════════════════════════════════════
@app.route('/api/personnel', methods=['GET','POST'])
def personnel_list():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_centre','chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        conn = get_db()
        email = generate_email(d['nom'], d.get('prenom',''), conn)
        password = gen_password()
        lid = conn.execute('''INSERT INTO personnel(nom,prenom,email,mot_de_passe,telephone,dateNaissance,
            fonction,role,adresse,superieur_id,bureau_id,annee_integration,actif,statut_emploi,photo)
            VALUES(?,?,?,?,?,?,?,?,?,?,?,?,1,"Actif",?)''',
            (d['nom'],d.get('prenom'),email,h(password),d.get('telephone'),d.get('dateNaissance'),
             d.get('fonction'),d['role'],d.get('adresse'),d.get('superieur_id') or None,
             d.get('bureau_id') or None,d.get('annee_integration',date.today().year),d.get('photo'))).lastrowid
        conn.commit(); conn.close()
        return jsonify({'personnel_id': lid, 'email': email, 'password': password}), 201
    # GET
    include_inactive = request.args.get('all') == '1' and r == 'chef_centre'
    actif_filter = '' if include_inactive else 'AND p.actif=1'
    if r == 'chef_centre':
        rows = qry(f'SELECT p.*,b.nom bureau_nom,s.nom sup_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel s ON p.superieur_id=s.personnel_id WHERE 1=1 {actif_filter} ORDER BY p.actif DESC,p.role,p.nom')
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        rows = qry(f'SELECT p.*,b.nom bureau_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.bureau_id=? {actif_filter} ORDER BY p.role,p.nom', (bid,))
    elif r == 'controleur':
        rows = qry(f'SELECT p.*,b.nom bureau_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.superieur_id=? {actif_filter}', (u,))
    else:
        rows = qry('SELECT p.*,b.nom bureau_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.personnel_id=?', (u,))
    for row in rows: row.pop('mot_de_passe', None)
    return jsonify(rows)

@app.route('/api/personnel/<int:pid>', methods=['GET','PUT','DELETE'])
def personnel_detail(pid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if request.method == 'GET':
        p = qry('SELECT p.*,b.nom bureau_nom,s.nom sup_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel s ON p.superieur_id=s.personnel_id WHERE p.personnel_id=?', (pid,), one=True)
        if p: p.pop('mot_de_passe', None)
        return jsonify(p)
    if role() not in ('chef_centre','chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
    if request.method == 'DELETE':
        # Désactiver au lieu de supprimer
        exe('UPDATE personnel SET actif=0 WHERE personnel_id=?', (pid,))
        notif('Compte désactivé', 'Votre accès à TERRAFISC a été désactivé par l\'administration.', 'alerte', pid)
        return jsonify({'ok': True})
    d = request.json
    exe('UPDATE personnel SET nom=?,prenom=?,telephone=?,fonction=?,role=?,adresse=?,superieur_id=?,bureau_id=?,photo=? WHERE personnel_id=?',
        (d.get('nom'),d.get('prenom'),d.get('telephone'),d.get('fonction'),d.get('role'),
         d.get('adresse'),d.get('superieur_id') or None,d.get('bureau_id') or None,d.get('photo'),pid))
    return jsonify({'ok': True})

@app.route('/api/personnel/<int:pid>/statut', methods=['PUT'])
def change_statut_emploi(pid):
    """N+1 peut changer le statut emploi de ses subordonnés (affecter, réaffecter, mise en pied)."""
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    target = qry('SELECT * FROM personnel WHERE personnel_id=? AND actif=1', (pid,), one=True)
    if not target: return jsonify({'error': 'Personnel introuvable'}), 404
    # Vérifier que c'est un subordonné direct ou chef_centre
    if r != 'chef_centre' and target.get('superieur_id') != u:
        return jsonify({'error': 'Non autorisé — ce personnel n\'est pas votre subordonné direct'}), 403
    d = request.json
    new_statut = d.get('statut_emploi', 'Actif')
    exe('UPDATE personnel SET statut_emploi=? WHERE personnel_id=?', (new_statut, pid))
    msgs = {'Actif':'Votre statut a été remis à Actif.','Réaffecté':'Vous avez été réaffecté(e) à un nouveau poste.',
            'Mis en pied':'Vous êtes temporairement mis(e) en pied. Contactez votre N+1.'}
    notif('Mise à jour de statut', msgs.get(new_statut, f'Statut changé : {new_statut}'), 'info', pid)
    return jsonify({'ok': True})

@app.route('/api/personnel/<int:pid>/reactiver', methods=['PUT'])
def reactiver_personnel(pid):
    if not auth() or role() != 'chef_centre': return jsonify({'error': 'Non autorisé'}), 403
    exe('UPDATE personnel SET actif=1,statut_emploi="Actif" WHERE personnel_id=?', (pid,))
    notif('Compte réactivé', 'Votre accès à TERRAFISC a été réactivé. Bienvenue !', 'info', pid)
    return jsonify({'ok': True})

# ═══════════════════════════════════════════════════════════════
#  ACTIVITÉS
# ═══════════════════════════════════════════════════════════════
@app.route('/api/activites', methods=['GET','POST'])
def activites():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_centre','chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        lid = exe('INSERT INTO activite(nom,type_activite,description,dateDebut,dateFin,statut,bureau_id,propose_par) VALUES(?,?,?,?,?,?,?,?)',
            (d['nom'],d.get('type_activite'),d.get('description'),d.get('dateDebut'),d.get('dateFin'),d.get('statut','Planifiee'),d.get('bureau_id'),u))
        return jsonify({'activite_id': lid}), 201
    base = 'SELECT a.*,b.nom bureau_nom,(SELECT COUNT(*) FROM tache WHERE activite_id=a.activite_id) nb_taches,(SELECT COUNT(*) FROM tache WHERE activite_id=a.activite_id AND statut="Terminee") nb_ok FROM activite a LEFT JOIN bureau b ON a.bureau_id=b.bureau_id'
    if r == 'chef_centre':
        return jsonify(qry(base + ' ORDER BY a.dateDebut DESC'))
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry(base + ' WHERE a.bureau_id=? ORDER BY a.dateDebut DESC', (bid,)))
    else:
        return jsonify(qry(base + ' JOIN tache t ON a.activite_id=t.activite_id JOIN affecter af ON t.tache_id=af.tache_id WHERE af.personnel_id=? AND af.actif=1 GROUP BY a.activite_id', (u,)))

@app.route('/api/activites/<int:aid>', methods=['PUT','DELETE'])
def activite_detail(aid):
    if not auth() or role() not in ('chef_centre','chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
    if request.method == 'DELETE':
        exe('DELETE FROM activite WHERE activite_id=?', (aid,)); return jsonify({'ok': True})
    d = request.json
    exe('UPDATE activite SET nom=?,type_activite=?,description=?,dateDebut=?,dateFin=?,statut=? WHERE activite_id=?',
        (d.get('nom'),d.get('type_activite'),d.get('description'),d.get('dateDebut'),d.get('dateFin'),d.get('statut'),aid))
    return jsonify({'ok': True})

@app.route('/api/activites/<int:aid>/gantt')
def gantt(aid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify({
        'activite': qry('SELECT * FROM activite WHERE activite_id=?', (aid,), one=True),
        'taches': qry('SELECT t.*,GROUP_CONCAT(p.nom||" "||COALESCE(p.prenom,"")) agents FROM tache t LEFT JOIN affecter af ON t.tache_id=af.tache_id AND af.actif=1 LEFT JOIN personnel p ON af.personnel_id=p.personnel_id WHERE t.activite_id=? GROUP BY t.tache_id ORDER BY t.dateDebut', (aid,))
    })

# ═══════════════════════════════════════════════════════════════
#  TÂCHES
# ═══════════════════════════════════════════════════════════════
@app.route('/api/taches', methods=['GET','POST'])
def taches():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_bureau','controleur'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        lid = exe('INSERT INTO tache(libelle,description,dateDebut,dateFin,statut,priorite,activite_id) VALUES(?,?,?,?,?,?,?)',
            (d['libelle'],d.get('description'),d.get('dateDebut'),d.get('dateFin'),d.get('statut','Non demarre'),d.get('priorite','Normale'),d.get('activite_id')))
        return jsonify({'tache_id': lid}), 201
    base = 'SELECT t.*,a.nom act_nom,GROUP_CONCAT(p.nom||" "||COALESCE(p.prenom,"")) agents FROM tache t LEFT JOIN activite a ON t.activite_id=a.activite_id LEFT JOIN affecter af ON t.tache_id=af.tache_id AND af.actif=1 LEFT JOIN personnel p ON af.personnel_id=p.personnel_id'
    if r == 'chef_centre':
        return jsonify(qry(base + ' GROUP BY t.tache_id ORDER BY t.dateFin'))
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry(base + ' WHERE a.bureau_id=? GROUP BY t.tache_id ORDER BY t.dateFin', (bid,)))
    elif r == 'controleur':
        return jsonify(qry(base + ' WHERE af.assigne_par=? OR af.personnel_id=? GROUP BY t.tache_id ORDER BY t.dateFin', (u,u)))
    else:
        return jsonify(qry('SELECT t.*,a.nom act_nom,af.role_affect FROM tache t JOIN affecter af ON t.tache_id=af.tache_id JOIN activite a ON t.activite_id=a.activite_id WHERE af.personnel_id=? AND af.actif=1 ORDER BY t.dateFin', (u,)))

@app.route('/api/taches/<int:tid>', methods=['PUT','DELETE'])
def tache_detail(tid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'DELETE':
        if r not in ('chef_bureau','controleur'): return jsonify({'error': 'Non autorisé'}), 403
        exe('DELETE FROM tache WHERE tache_id=?', (tid,)); return jsonify({'ok': True})
    d = request.json; old = qry('SELECT * FROM tache WHERE tache_id=?', (tid,), one=True)
    if r == 'agent':
        exe('UPDATE tache SET statut=? WHERE tache_id=?', (d.get('statut',old['statut']), tid))
    else:
        exe('UPDATE tache SET libelle=?,description=?,dateDebut=?,dateFin=?,statut=?,priorite=? WHERE tache_id=?',
            (d.get('libelle',old['libelle']),d.get('description',old['description']),d.get('dateDebut',old['dateDebut']),d.get('dateFin',old['dateFin']),d.get('statut',old['statut']),d.get('priorite',old['priorite']),tid))
    if d.get('statut') and d['statut'] != old.get('statut'):
        for ag in qry('SELECT personnel_id FROM affecter WHERE tache_id=? AND actif=1', (tid,)):
            notif('Mise à jour tâche', f'Tâche "{old["libelle"]}" → {d["statut"]}', 'tache', ag['personnel_id'], tid)
    return jsonify({'ok': True})

@app.route('/api/taches/<int:tid>/affecter', methods=['POST'])
def affecter(tid):
    if not auth() or role() not in ('chef_bureau','controleur'): return jsonify({'error': 'Non autorisé'}), 403
    u = uid(); d = request.json; dest = int(d['personnel_id'])
    subs = [s['personnel_id'] for s in qry('SELECT personnel_id FROM personnel WHERE superieur_id=? AND actif=1', (u,))]
    if dest not in subs: return jsonify({'error': 'Vous ne pouvez affecter qu\'à vos subordonnés directs'}), 403
    exe('INSERT INTO affecter(tache_id,personnel_id,role_affect,date_affectation,actif,assigne_par) VALUES(?,?,?,?,1,?)',
        (tid, dest, d.get('role','Executant'), str(date.today()), u))
    tl = qry('SELECT libelle FROM tache WHERE tache_id=?', (tid,), one=True)
    notif('Nouvelle tâche assignée', f'Vous avez été affecté(e) à : "{tl["libelle"]}".', 'tache', dest, tid)
    return jsonify({'ok': True})

@app.route('/api/taches/<int:tid>/retirer/<int:pid>', methods=['DELETE'])
def retirer(tid, pid):
    if not auth() or role() not in ('chef_bureau','controleur'): return jsonify({'error': 'Non autorisé'}), 403
    exe('UPDATE affecter SET actif=0,date_retrait=? WHERE tache_id=? AND personnel_id=?', (str(date.today()), tid, pid))
    notif('Retrait d\'affectation', 'Vous avez été retiré(e) d\'une tâche.', 'info', pid, tid)
    return jsonify({'ok': True})

@app.route('/api/taches/<int:tid>/historique')
def historique_tache(tid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify(qry('SELECT af.*,p.nom||" "||COALESCE(p.prenom,"") agent_nom,s.nom||" "||COALESCE(s.prenom,"") sup_nom FROM affecter af JOIN personnel p ON af.personnel_id=p.personnel_id LEFT JOIN personnel s ON af.assigne_par=s.personnel_id WHERE af.tache_id=? ORDER BY af.date_affectation DESC', (tid,)))

# ═══════════════════════════════════════════════════════════════
#  PERFORMANCES
# ═══════════════════════════════════════════════════════════════
@app.route('/api/performances', methods=['GET','POST'])
def performances():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_bureau','controleur'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json; dest = int(d['personnel_id'])
        subs = [s['personnel_id'] for s in qry('SELECT personnel_id FROM personnel WHERE superieur_id=?', (u,))]
        if dest not in subs: return jsonify({'error': 'Vous ne pouvez évaluer que vos subordonnés'}), 403
        m = date.today().month; an = date.today().year
        lid = exe('INSERT INTO performance(efficacite,note,prime,commentaire,personnel_id,evalue_par,date_eval,mois,annee) VALUES(?,?,?,?,?,?,?,?,?)',
            (d.get('efficacite'),d.get('note'),d.get('prime'),d.get('commentaire'),dest,u,str(date.today()),m,an))
        notif('Nouvelle évaluation', f'Vous avez reçu une évaluation pour {MOIS_FR[m]} {an}.', 'performance', dest)
        return jsonify({'performance_id': lid}), 201
    if r == 'chef_centre':
        return jsonify(qry('SELECT pf.*,p.nom,p.prenom,b.nom bureau_nom,e.nom eval_nom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel e ON pf.evalue_par=e.personnel_id ORDER BY pf.date_eval DESC'))
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry('SELECT pf.*,p.nom,p.prenom,e.nom eval_nom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id LEFT JOIN personnel e ON pf.evalue_par=e.personnel_id WHERE p.bureau_id=? ORDER BY pf.date_eval DESC', (bid,)))
    elif r == 'controleur':
        return jsonify(qry('SELECT pf.*,p.nom,p.prenom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id WHERE pf.evalue_par=? ORDER BY pf.date_eval DESC', (u,)))
    else:
        return jsonify(qry('SELECT pf.*,e.nom eval_nom FROM performance pf LEFT JOIN personnel e ON pf.evalue_par=e.personnel_id WHERE pf.personnel_id=? ORDER BY pf.date_eval DESC', (u,)))

# ═══════════════════════════════════════════════════════════════
#  EMPLOYÉ DU MOIS
# ═══════════════════════════════════════════════════════════════
@app.route('/api/employe-mois', methods=['GET','POST'])
def employe_mois():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if request.method == 'POST':
        if role() not in ('chef_centre','chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        m = int(d.get('mois', date.today().month)); an = int(d.get('annee', date.today().year))
        ex = qry('SELECT em_id FROM employe_mois WHERE mois=? AND annee=?', (m,an), one=True)
        if ex:
            exe('UPDATE employe_mois SET personnel_id=?,note_finale=?,motif=?,designe_par=?,date_designation=? WHERE mois=? AND annee=?',
                (d['personnel_id'],d.get('note_finale'),d.get('motif'),uid(),str(date.today()),m,an))
        else:
            exe('INSERT INTO employe_mois(personnel_id,mois,annee,note_finale,motif,designe_par,date_designation) VALUES(?,?,?,?,?,?,?)',
                (d['personnel_id'],m,an,d.get('note_finale'),d.get('motif'),uid(),str(date.today())))
        notif('🏆 Employé du mois !', f'Félicitations ! Vous êtes l\'Employé du mois de {MOIS_FR[m]} {an}.', 'employe_mois', int(d['personnel_id']))
        return jsonify({'ok': True}), 201
    return jsonify(qry('SELECT em.*,p.nom,p.prenom,p.fonction,b.nom bureau_nom,d.nom designateur_nom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel d ON em.designe_par=d.personnel_id ORDER BY em.annee DESC,em.mois DESC'))

# ═══════════════════════════════════════════════════════════════
#  TÉLÉPAIEMENT
# ═══════════════════════════════════════════════════════════════
@app.route('/api/telepaiements', methods=['GET','POST'])
def telepaiements():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        d = request.json
        cnt = qry('SELECT COUNT(*) c FROM telepaiement', one=True)['c']
        ref = f'TP{date.today().year}-{str(cnt+1).zfill(3)}'
        lid = exe('INSERT INTO telepaiement(reference,contribuable_nom,contribuable_nif,type_impot,montant,montant_paye,statut,mode_paiement,date_echeance,bureau_id,agent_id,notes) VALUES(?,?,?,?,?,?,?,?,?,?,?,?)',
            (ref,d['contribuable_nom'],d.get('contribuable_nif'),d['type_impot'],d['montant'],d.get('montant_paye',0),d.get('statut','En attente'),d.get('mode_paiement'),d.get('date_echeance'),d.get('bureau_id'),u,d.get('notes')))
        return jsonify({'tp_id': lid, 'reference': ref}), 201
    base = 'SELECT tp.*,b.nom bureau_nom,p.nom agent_nom,p.prenom agent_prenom FROM telepaiement tp LEFT JOIN bureau b ON tp.bureau_id=b.bureau_id LEFT JOIN personnel p ON tp.agent_id=p.personnel_id'
    if r == 'chef_centre':
        return jsonify(qry(base + ' ORDER BY tp.date_echeance'))
    bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
    return jsonify(qry(base + ' WHERE tp.bureau_id=? ORDER BY tp.date_echeance', (bid,)))

@app.route('/api/telepaiements/<int:tid>', methods=['PUT'])
def telepaiement_update(tid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    d = request.json
    exe('UPDATE telepaiement SET statut=?,montant_paye=?,mode_paiement=?,date_paiement=?,notes=? WHERE tp_id=?',
        (d.get('statut'),d.get('montant_paye'),d.get('mode_paiement'),d.get('date_paiement',str(date.today())),d.get('notes'),tid))
    return jsonify({'ok': True})

@app.route('/api/telepaiements/stats')
def tp_stats():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify({
        'par_statut': qry('SELECT statut,COUNT(*) n,ROUND(SUM(montant)/1000000,2) total_m FROM telepaiement GROUP BY statut'),
        'par_bureau': qry('SELECT b.nom,COUNT(*) n,ROUND(SUM(tp.montant)/1000000,2) total_m,ROUND(SUM(tp.montant_paye)/1000000,2) paye_m FROM telepaiement tp JOIN bureau b ON tp.bureau_id=b.bureau_id GROUP BY b.nom'),
        'par_type': qry('SELECT type_impot,COUNT(*) n,ROUND(SUM(montant)/1000000,2) total_m FROM telepaiement GROUP BY type_impot ORDER BY total_m DESC'),
        'total_attendu': (qry('SELECT ROUND(SUM(montant)/1000000,2) v FROM telepaiement', one=True) or {}).get('v') or 0,
        'total_recouvre': (qry('SELECT ROUND(SUM(montant_paye)/1000000,2) v FROM telepaiement', one=True) or {}).get('v') or 0
    })

# ═══════════════════════════════════════════════════════════════
#  NOTIFICATIONS, SIGNALEMENTS, PROPOSITIONS, AVIS, CR
# ═══════════════════════════════════════════════════════════════
@app.route('/api/notifications')
def notifications():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify(qry('SELECT n.*,t.libelle tache_libelle FROM notification n LEFT JOIN tache t ON n.tache_id=t.tache_id WHERE n.destinataire_id=? ORDER BY n.dateEnvoie DESC', (uid(),)))

@app.route('/api/notifications/<int:nid>/lire', methods=['PUT'])
def lire_notif(nid):
    exe("UPDATE notification SET statut='Lue' WHERE notification_id=?", (nid,)); return jsonify({'ok': True})

@app.route('/api/notifications/tout-lire', methods=['PUT'])
def tout_lire():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    exe("UPDATE notification SET statut='Lue' WHERE destinataire_id=?", (uid(),)); return jsonify({'ok': True})

@app.route('/api/signalements', methods=['GET','POST'])
def signalements():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        d = request.json
        sup = qry('SELECT superieur_id FROM personnel WHERE personnel_id=?', (u,), one=True)['superieur_id']
        lid = exe('INSERT INTO signalement(description,dateEnvoie,statut,personnel_id,tache_id,destine_a) VALUES(?,?,?,?,?,?)',
            (d['description'], str(date.today()), 'Ouvert', u, d.get('tache_id'), sup))
        if sup: notif('Nouveau signalement', 'Un collaborateur a émis un signalement.', 'alerte', sup)
        return jsonify({'signalement_id': lid}), 201
    if r == 'chef_centre':
        return jsonify(qry('SELECT s.*,p.nom auteur,t.libelle tache_lib FROM signalement s LEFT JOIN personnel p ON s.personnel_id=p.personnel_id LEFT JOIN tache t ON s.tache_id=t.tache_id ORDER BY s.dateEnvoie DESC'))
    elif r in ('chef_bureau','controleur'):
        return jsonify(qry('SELECT s.*,p.nom auteur,t.libelle tache_lib FROM signalement s LEFT JOIN personnel p ON s.personnel_id=p.personnel_id LEFT JOIN tache t ON s.tache_id=t.tache_id WHERE s.destine_a=? ORDER BY s.dateEnvoie DESC', (u,)))
    else:
        return jsonify(qry('SELECT s.*,t.libelle tache_lib FROM signalement s LEFT JOIN tache t ON s.tache_id=t.tache_id WHERE s.personnel_id=? ORDER BY s.dateEnvoie DESC', (u,)))

@app.route('/api/signalements/<int:sid>/repondre', methods=['PUT'])
def repondre_signal(sid):
    d = request.json
    exe('UPDATE signalement SET reponse=?,statut=? WHERE signalement_id=?', (d.get('reponse'), d.get('statut','Traite'), sid))
    sig = qry('SELECT personnel_id FROM signalement WHERE signalement_id=?', (sid,), one=True)
    notif('Signalement traité', 'Votre signalement a reçu une réponse.', 'info', sig['personnel_id'])
    return jsonify({'ok': True})

@app.route('/api/propositions', methods=['GET','POST'])
def propositions():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('controleur','agent'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        sup = qry('SELECT superieur_id FROM personnel WHERE personnel_id=?', (u,), one=True)['superieur_id']
        if not sup: return jsonify({'error': 'Pas de supérieur défini'}), 400
        lid = exe('INSERT INTO proposition_activite(description,type_activite,date_prop,propose_par,destine_a) VALUES(?,?,?,?,?)',
            (d['description'], d.get('type_activite','Mission'), str(date.today()), u, sup))
        notif('Nouvelle proposition d\'activité', 'Un collaborateur vous a soumis une proposition.', 'proposition', sup)
        return jsonify({'prop_id': lid}), 201
    if r in ('chef_centre','chef_bureau'):
        return jsonify(qry('SELECT pa.*,p.nom proposeur,p.prenom proposeur_prenom FROM proposition_activite pa JOIN personnel p ON pa.propose_par=p.personnel_id WHERE pa.destine_a=? ORDER BY pa.date_prop DESC', (u,)))
    else:
        return jsonify(qry('SELECT pa.*,p.nom destinataire FROM proposition_activite pa JOIN personnel p ON pa.destine_a=p.personnel_id WHERE pa.propose_par=? ORDER BY pa.date_prop DESC', (u,)))

@app.route('/api/propositions/<int:pid>/repondre', methods=['PUT'])
def repondre_prop(pid):
    d = request.json
    exe('UPDATE proposition_activite SET statut=?,commentaire_reponse=? WHERE prop_id=?', (d.get('statut'), d.get('commentaire'), pid))
    prop = qry('SELECT propose_par FROM proposition_activite WHERE prop_id=?', (pid,), one=True)
    notif(f'Proposition {d.get("statut","").lower()}', f'Votre proposition a été {d.get("statut","").lower()}.', 'info', prop['propose_par'])
    return jsonify({'ok': True})

@app.route('/api/avis', methods=['GET','POST'])
def avis():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if request.method == 'POST':
        d = request.json
        exe('INSERT INTO avis(commentaire,note,dateEnvoie,personnel_id) VALUES(?,?,?,?)', (d.get('commentaire'),d.get('note'),str(date.today()),uid()))
        return jsonify({'ok': True}), 201
    return jsonify(qry('SELECT av.*,p.nom,p.prenom,b.nom bureau_nom FROM avis av LEFT JOIN personnel p ON av.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id ORDER BY av.dateEnvoie DESC'))

@app.route('/api/comptes-rendus', methods=['GET','POST'])
def comptes_rendus():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r == 'chef_centre':
            return jsonify({'error': 'Le chef de centre ne soumet pas de comptes-rendus'}), 403
        d = request.json
        exe('INSERT INTO compteRendu(dateRendue,contenu,statut,personnel_id,tache_id) VALUES(?,?,?,?,?)',
            (str(date.today()), d['contenu'], d.get('statut','Soumis'), u, d.get('tache_id')))
        sup = qry('SELECT superieur_id FROM personnel WHERE personnel_id=?', (u,), one=True)['superieur_id']
        if sup: notif('Nouveau compte-rendu', 'Un collaborateur a soumis un compte-rendu.', 'info', sup)
        return jsonify({'ok': True}), 201
    if r in ('chef_bureau','controleur'):
        return jsonify(qry('SELECT cr.*,p.nom,p.prenom,t.libelle tache_lib FROM compteRendu cr LEFT JOIN personnel p ON cr.personnel_id=p.personnel_id LEFT JOIN tache t ON cr.tache_id=t.tache_id ORDER BY cr.dateRendue DESC'))
    else:
        return jsonify(qry('SELECT cr.*,t.libelle tache_lib FROM compteRendu cr LEFT JOIN tache t ON cr.tache_id=t.tache_id WHERE cr.personnel_id=? ORDER BY cr.dateRendue DESC', (u,)))

# ═══════════════════════════════════════════════════════════════
#  RÉINITIALISATION (chef_centre uniquement)
# ═══════════════════════════════════════════════════════════════
@app.route('/api/reset', methods=['POST'])
def reset_app():
    if not auth() or role() != 'chef_centre':
        return jsonify({'error': 'Non autorisé'}), 403
    conn = get_db()
    conn.executescript('''
        DELETE FROM avis;
        DELETE FROM compteRendu;
        DELETE FROM signalement;
        DELETE FROM proposition_activite;
        DELETE FROM notification;
        DELETE FROM affecter;
        DELETE FROM performance;
        DELETE FROM employe_mois;
        DELETE FROM tache;
        DELETE FROM activite;
        DELETE FROM telepaiement;
        DELETE FROM personnel;
        DELETE FROM bureau;
    ''')
    try:
        conn.execute("UPDATE sqlite_sequence SET seq=0")
    except:
        pass
    admin_pwd = 'TF@Admin2025'
    conn.execute("""INSERT INTO personnel(nom,prenom,email,mot_de_passe,fonction,role,actif,annee_integration)
        VALUES(?,?,?,?,?,?,1,?)""",
        ('ADMIN','Chef','admin@terrafisc.sn',h(admin_pwd),'Directeur du Centre','chef_centre',date.today().year))
    conn.commit(); conn.close()
    session.clear()
    return jsonify({'ok': True, 'email': 'admin@terrafisc.sn', 'password': admin_pwd})

print('✅ Application Flask TERRAFISC prête')
print(f'   {len(list(app.url_map.iter_rules()))} routes définies')


✅ Application Flask TERRAFISC prête
   33 routes définies


## 🎨 Cellule 4 — Interface HTML / CSS / JS

In [ ]:
HTML_TEMPLATE = r"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>TF · TERRAFISC</title>
<link href="https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700;800&family=Lato:wght@300;400;700;900&display=swap" rel="stylesheet">
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
:root{
  --br:#5C2E0E;--brd:#3D1A07;--brl:#8B5A3A;--brp:#F2EAE2;
  --gd:#C9A840;--gdl:#E8C96A;--gdp:#FBF5E8;
  --wh:#FFFFFF;--cr:#FAF7F2;--cr2:#F5EEE5;
  --tx:#2D1A0E;--mt:#9E7B64;--bd:#E8D5C4;
  --ok:#2E7D52;--er:#B71C1C;--wn:#C17B1A;--in:#1565C0;
  --sh:0 2px 12px rgba(92,46,14,.12);
  --sh2:0 8px 32px rgba(92,46,14,.2);
}
*{box-sizing:border-box;margin:0;padding:0}
html,body{height:100%}
body{font-family:'Lato',sans-serif;background:var(--cr);color:var(--tx);font-size:14px}

/* ── LOGIN ── */
#login-screen{min-height:100vh;display:flex;align-items:center;justify-content:center;background:linear-gradient(145deg,var(--brd) 0%,var(--br) 55%,#8B4513 100%);position:relative;overflow:hidden}
#login-screen::before{content:'TF';position:absolute;font-family:'Playfair Display',serif;font-size:28rem;font-weight:800;color:rgba(201,168,64,.06);right:-6rem;top:-4rem;line-height:1;pointer-events:none}
.lbox{background:var(--wh);border-radius:20px;padding:48px 44px;width:420px;box-shadow:var(--sh2);position:relative;z-index:1}
.tf-logo{display:flex;align-items:center;justify-content:center;gap:12px;margin-bottom:8px}
.tf-mark{width:56px;height:56px;background:var(--br);border-radius:12px;display:flex;align-items:center;justify-content:center;font-family:'Playfair Display',serif;font-size:1.6rem;font-weight:800;color:var(--gd);letter-spacing:-1px;box-shadow:0 4px 16px rgba(92,46,14,.35)}
.tf-wordmark{font-family:'Playfair Display',serif;font-size:1.8rem;font-weight:800;color:var(--br);letter-spacing:1px}
.tf-wordmark span{color:var(--gd)}
.tf-sub{text-align:center;font-size:.78rem;color:var(--mt);margin-bottom:32px;letter-spacing:.5px;font-weight:300}
.fg{margin-bottom:14px}
.fg label{display:block;font-size:.73rem;font-weight:700;color:var(--mt);margin-bottom:5px;text-transform:uppercase;letter-spacing:.6px}
input,select,textarea{width:100%;padding:10px 13px;border:1.5px solid var(--bd);border-radius:9px;font-size:.88rem;color:var(--tx);background:var(--wh);outline:none;transition:border .13s;font-family:'Lato',sans-serif}
input:focus,select:focus,textarea:focus{border-color:var(--gd);box-shadow:0 0 0 3px rgba(201,168,64,.12)}
.btn-login{width:100%;padding:12px;background:var(--br);color:var(--wh);border:none;border-radius:10px;cursor:pointer;font-size:.95rem;font-weight:700;font-family:'Lato',sans-serif;letter-spacing:.3px;transition:.15s;margin-top:6px}
.btn-login:hover{background:var(--brd)}

/* ── APP SHELL ── */
#app{display:none;min-height:100vh;flex-direction:column}
header{background:var(--br);color:var(--wh);height:56px;display:flex;align-items:center;padding:0 22px;gap:14px;position:sticky;top:0;z-index:300;box-shadow:0 3px 12px rgba(0,0,0,.3)}
.hlogo{display:flex;align-items:center;gap:9px}
.hmark{width:34px;height:34px;background:var(--gd);border-radius:8px;display:flex;align-items:center;justify-content:center;font-family:'Playfair Display',serif;font-size:1rem;font-weight:800;color:var(--br);flex-shrink:0}
.hwm{font-family:'Playfair Display',serif;font-size:1.1rem;font-weight:800;color:var(--wh);letter-spacing:1px}
.hwm span{color:var(--gdl)}
#rbadge{background:rgba(201,168,64,.18);border:1px solid rgba(201,168,64,.35);border-radius:20px;padding:3px 11px;font-size:.68rem;font-weight:700;text-transform:uppercase;letter-spacing:.7px;color:var(--gdl)}
.hright{margin-left:auto;display:flex;align-items:center;gap:12px}
#uname{font-size:.83rem;opacity:.9;color:rgba(255,255,255,.85)}
#notif-btn{cursor:pointer;background:none;border:none;color:var(--wh);font-size:1.1rem;position:relative;padding:4px}
#nc{position:absolute;top:-5px;right:-5px;background:var(--er);color:#fff;border-radius:50%;width:17px;height:17px;font-size:.6rem;display:none;align-items:center;justify-content:center;font-weight:700}
.hbtn{background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.22);color:var(--wh);padding:6px 13px;border-radius:8px;cursor:pointer;font-size:.78rem;font-family:'Lato',sans-serif;transition:.13s}
.hbtn:hover{background:rgba(255,255,255,.2)}

/* ── LAYOUT ── */
.layout{display:flex;min-height:calc(100vh - 56px)}
nav{width:220px;background:var(--wh);border-right:1px solid var(--bd);padding:16px 0;flex-shrink:0;position:sticky;top:56px;height:calc(100vh - 56px);overflow-y:auto}
.nsec{padding:10px 18px 4px;font-size:.62rem;font-weight:900;text-transform:uppercase;letter-spacing:1.2px;color:var(--mt)}
.ni{display:flex;align-items:center;gap:9px;padding:9px 18px 9px 16px;cursor:pointer;color:var(--tx);font-size:.84rem;border-left:3px solid transparent;transition:.12s;user-select:none}
.ni:hover{background:var(--gdp);color:var(--br)}
.ni.active{background:var(--brp);color:var(--br);border-left-color:var(--gd);font-weight:700}
.nico{font-size:14px;width:20px;text-align:center;flex-shrink:0}
.main{flex:1;padding:24px;overflow-y:auto;background:var(--cr)}

/* ── PAGES ── */
.page{display:none}.page.active{display:block}
.ptitle{font-family:'Playfair Display',serif;font-size:1.3rem;font-weight:700;color:var(--br);margin-bottom:20px;display:flex;align-items:center;gap:12px;flex-wrap:wrap;border-bottom:2px solid var(--bd);padding-bottom:14px}
.ptitle-ico{width:36px;height:36px;background:var(--gdp);border-radius:9px;display:flex;align-items:center;justify-content:center;font-size:1rem}

/* ── STATS ── */
.gstats{display:grid;grid-template-columns:repeat(auto-fill,minmax(148px,1fr));gap:12px;margin-bottom:22px}
.stat{background:var(--wh);border-radius:12px;padding:16px;border:1px solid var(--bd);border-top:3px solid var(--br);transition:.15s}
.stat:hover{transform:translateY(-2px);box-shadow:var(--sh)}
.stat .n{font-family:'Playfair Display',serif;font-size:1.9rem;font-weight:700;color:var(--br);line-height:1}
.stat .l{font-size:.72rem;color:var(--mt);margin-top:4px;font-weight:700;text-transform:uppercase;letter-spacing:.4px}
.stat.gd{border-top-color:var(--gd)}.stat.gd .n{color:var(--gd)}
.stat.ok{border-top-color:var(--ok)}.stat.ok .n{color:var(--ok)}
.stat.wn{border-top-color:var(--wn)}.stat.wn .n{color:var(--wn)}
.stat.er{border-top-color:var(--er)}.stat.er .n{color:var(--er)}

/* ── CARDS ── */
.card{background:var(--wh);border-radius:12px;border:1px solid var(--bd);margin-bottom:18px;overflow:hidden}
.chead{padding:14px 20px;border-bottom:1px solid var(--bd);display:flex;align-items:center;justify-content:space-between;background:linear-gradient(to right,var(--cr),var(--wh))}
.chead h3{font-family:'Playfair Display',serif;font-size:.95rem;font-weight:700;color:var(--br)}
.cbody{padding:16px 20px}

/* ── TABLE ── */
table{width:100%;border-collapse:collapse}
th{background:var(--gdp);color:var(--br);font-size:.7rem;font-weight:900;text-transform:uppercase;letter-spacing:.6px;padding:11px 13px;text-align:left;border-bottom:2px solid var(--gd)}
td{padding:11px 13px;border-bottom:1px solid var(--cr2);font-size:.83rem;vertical-align:middle}
tr:last-child td{border:none}
tr:hover td{background:var(--gdp)}

/* ── BUTTONS ── */
.btn{padding:7px 14px;border:none;border-radius:8px;cursor:pointer;font-size:.78rem;font-weight:700;font-family:'Lato',sans-serif;transition:.13s;display:inline-flex;align-items:center;gap:5px;white-space:nowrap}
.bbr{background:var(--br);color:#fff}.bbr:hover{background:var(--brd)}
.bgd{background:var(--gd);color:var(--br)}.bgd:hover{background:var(--gdl)}
.brd{background:var(--er);color:#fff;padding:4px 9px;font-size:.72rem}
.bok{background:var(--ok);color:#fff}
.bwn{background:var(--wn);color:#fff}
.bo{background:transparent;border:1.5px solid var(--bd);color:var(--tx)}.bo:hover{background:var(--cr2)}
.bsm{padding:4px 9px;font-size:.72rem}

/* ── BADGES ── */
.badge{display:inline-flex;align-items:center;padding:3px 8px;border-radius:20px;font-size:.68rem;font-weight:700;white-space:nowrap;letter-spacing:.3px}
.bs{background:#C8F5E0;color:#0B5E36}.bw{background:#FFF3CD;color:#7A5200}.be{background:#FDE8E8;color:#8B1919}.bi{background:#D3E5FB;color:#1254A0}.bm{background:#F0EBE4;color:#6B5040}.bgdl{background:#FBF5E8;color:#7A5200}.bp2{background:#EDE7F6;color:#4A2FAA}
.rc{display:inline-flex;padding:2px 8px;border-radius:20px;font-size:.66rem;font-weight:900;text-transform:uppercase;letter-spacing:.5px}
.rc-cc{background:var(--br);color:var(--gd)}.rc-cb{background:#145c3a;color:#fff}.rc-ctrl{background:#7a4a08;color:#fff}.rc-agent{background:#3d3d3d;color:#fff}

/* ── FORMS ── */
.fg{margin-bottom:14px}.fg label{display:block;font-size:.72rem;font-weight:700;color:var(--mt);margin-bottom:5px;text-transform:uppercase;letter-spacing:.6px}
.fr{display:grid;grid-template-columns:1fr 1fr;gap:14px}

/* ── MODALS ── */
.mo{display:none;position:fixed;inset:0;background:rgba(61,26,7,.6);z-index:1000;overflow-y:auto;padding:28px 14px;align-items:flex-start;justify-content:center}
.mo.open{display:flex}
.mobox{background:var(--wh);border-radius:16px;padding:28px;width:100%;max-width:520px;margin:auto;box-shadow:var(--sh2)}
.mobox h3{font-family:'Playfair Display',serif;font-size:1.1rem;font-weight:700;color:var(--br);margin-bottom:20px;padding-bottom:12px;border-bottom:2px solid var(--gdp)}
.moftr{display:flex;gap:10px;justify-content:flex-end;margin-top:20px;padding-top:16px;border-top:1px solid var(--bd)}

/* ── NOTIF PANEL ── */
.np{position:fixed;top:56px;right:0;width:340px;background:var(--wh);border-left:1px solid var(--bd);height:calc(100vh - 56px);overflow-y:auto;z-index:300;transform:translateX(100%);transition:.22s;box-shadow:-4px 0 24px rgba(92,46,14,.12)}
.np.open{transform:none}
.nphead{padding:14px 18px;border-bottom:2px solid var(--gd);display:flex;align-items:center;justify-content:space-between;background:var(--gdp)}
.nitem{padding:12px 16px;border-bottom:1px solid var(--cr2);cursor:pointer;transition:.1s}
.nitem:hover{background:var(--gdp)}
.nitem.unread{background:var(--brp);border-left:3px solid var(--gd)}
.ntit{font-weight:700;font-size:.8rem;color:var(--br);margin-bottom:2px}
.ndesc{font-size:.75rem;color:var(--mt);line-height:1.4}
.ntime{font-size:.67rem;color:var(--mt);margin-top:3px}

/* ── EMPLOYE MOIS ── */
.em-card{background:linear-gradient(135deg,var(--br),#8B4513);color:#fff;border-radius:14px;padding:22px;margin-bottom:14px;position:relative;overflow:hidden;box-shadow:var(--sh)}
.em-card::after{content:'🏆';position:absolute;right:18px;top:50%;transform:translateY(-50%);font-size:3.2rem;opacity:.25}
.em-month{font-size:.72rem;opacity:.8;text-transform:uppercase;letter-spacing:.8px;font-weight:700}
.em-name{font-family:'Playfair Display',serif;font-size:1.4rem;font-weight:700;margin:5px 0;color:var(--gdl)}
.em-bureau{font-size:.8rem;opacity:.85}
.em-note{font-size:.82rem;margin-top:8px;opacity:.9}

/* ── PROFIL PHOTO ── */
.photo-wrap{position:relative;width:100px;height:100px;margin:0 auto 16px}
.photo-wrap img{width:100%;height:100%;border-radius:50%;object-fit:cover;border:3px solid var(--gd)}
.photo-overlay{position:absolute;inset:0;border-radius:50%;background:rgba(92,46,14,.5);display:none;align-items:center;justify-content:center;cursor:pointer;color:#fff;font-size:.72rem;font-weight:700}
.photo-wrap:hover .photo-overlay{display:flex}
.avatar{width:36px;height:36px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:.74rem;font-weight:800;flex-shrink:0;font-family:'Playfair Display',serif}

/* ── MISC ── */
.cht{position:relative;height:195px}
.g2{display:grid;grid-template-columns:1fr 1fr;gap:16px}
.g3{display:grid;grid-template-columns:1fr 1fr 1fr;gap:16px}
.pgbar{background:var(--cr2);border-radius:6px;height:7px;overflow:hidden;margin-top:4px}
.pgfill{height:100%;border-radius:6px;background:var(--gd);transition:width .3s}
.empty{text-align:center;padding:40px 20px;color:var(--mt)}
.empty .eico{font-size:2.5rem;margin-bottom:8px}
.acts{display:flex;gap:5px;flex-wrap:wrap}
.prio-H td:first-child{border-left:3px solid var(--er)}
.prio-U td:first-child{border-left:3px solid #6F42C1}
.cred-box{background:var(--gdp);border:2px solid var(--gd);border-radius:12px;padding:18px;margin:14px 0;font-family:monospace}
.cred-box .c-row{display:flex;justify-content:space-between;padding:5px 0;font-size:.88rem}
.cred-box .c-row b{color:var(--br)}
.cred-box .c-row span{color:var(--brd);font-weight:700}
.statut-actif{color:var(--ok);font-weight:700}
.statut-pied{color:var(--er);font-weight:700}
.statut-reaffect{color:var(--wn);font-weight:700}
.sep{height:1px;background:var(--bd);margin:12px 0}
#toast{position:fixed;bottom:24px;right:24px;color:#fff;padding:12px 18px;border-radius:10px;display:none;z-index:9999;font-size:.84rem;max-width:310px;box-shadow:var(--sh2);font-weight:600}
@media(max-width:720px){nav{display:none}.g2,.g3{grid-template-columns:1fr}.fr{grid-template-columns:1fr}}
</style>
</head>
"""

HTML_BODY = r"""
<body>

<!-- ═══════════════════ LOGIN ═══════════════════ -->
<div id="login-screen">
 <div class="lbox">
  <div class="tf-logo">
   <div class="tf-mark">TF</div>
   <div class="tf-wordmark">TERRA<span>FISC</span></div>
  </div>
  <div class="tf-sub">Système de Gestion Administrative & Fiscale</div>
  <div id="lerr" style="background:#FDE8E8;color:#8B1919;padding:9px 13px;border-radius:8px;font-size:.8rem;margin-bottom:14px;display:none;border-left:3px solid var(--er)"></div>
  <div class="fg"><label>Email</label><input id="l-e" type="email" placeholder="votre.email@terrafisc.sn"></div>
  <div class="fg"><label>Mot de passe</label><input id="l-p" type="password" placeholder="••••••••"></div>
  <button class="btn-login" onclick="doLogin()">Se connecter</button>
 </div>
</div>

<!-- ═══════════════════ APP ═══════════════════ -->
<div id="app" style="display:none;flex-direction:column">
 <header>
  <div class="hlogo">
   <div class="hmark">TF</div>
   <div class="hwm">TERRA<span>FISC</span></div>
  </div>
  <div id="rbadge"></div>
  <div class="hright">
   <span id="uname"></span>
   <button id="notif-btn" onclick="toggleNP()" title="Notifications">🔔<span id="nc">0</span></button>
   <button class="hbtn" onclick="showPage('profil')">👤 Profil</button>
   <button class="hbtn" onclick="doLogout()">Déconnexion</button>
  </div>
 </header>
 <div class="layout">
  <nav id="nav"></nav>
  <div class="main">
   <div class="page active" id="pg-dashboard"></div>
   <div class="page" id="pg-bureaux"></div>
   <div class="page" id="pg-activites"></div>
   <div class="page" id="pg-taches"></div>
   <div class="page" id="pg-affecter"></div>
   <div class="page" id="pg-personnel"></div>
   <div class="page" id="pg-performances"></div>
   <div class="page" id="pg-employe-mois"></div>
   <div class="page" id="pg-telepaiement"></div>
   <div class="page" id="pg-signalements"></div>
   <div class="page" id="pg-propositions"></div>
   <div class="page" id="pg-cr"></div>
   <div class="page" id="pg-avis"></div>
   <div class="page" id="pg-profil"></div>
  </div>
 </div>
 <div class="np" id="np">
  <div class="nphead"><b style="font-size:.9rem;color:var(--br)">🔔 Notifications</b><button class="btn bo bsm" onclick="toutLire()">Tout lire</button></div>
  <div id="nlist"></div>
 </div>
</div>

<!-- ═══════════════════ MODALS ═══════════════════ -->

<!-- ACTIVITÉ -->
<div class="mo" id="mo-activite"><div class="mobox">
 <h3>📋 Nouvelle activité</h3>
 <div class="fr"><div class="fg"><label>Nom</label><input id="ma-n"></div><div class="fg"><label>Type</label><select id="ma-t"><option>Mission</option><option>Atelier</option><option>Séminaire</option><option>Forum</option><option>Colloque</option><option>Audit</option><option>Formation</option><option>Projet</option></select></div></div>
 <div class="fr"><div class="fg"><label>Date début</label><input type="date" id="ma-d"></div><div class="fg"><label>Date fin</label><input type="date" id="ma-f"></div></div>
 <div class="fg"><label>Bureau</label><select id="ma-b"></select></div>
 <div class="fg"><label>Description</label><textarea id="ma-desc" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-activite')">Annuler</button><button class="btn bbr" onclick="saveActivite()">Enregistrer</button></div>
</div></div>

<!-- TÂCHE -->
<div class="mo" id="mo-tache"><div class="mobox">
 <h3>✅ Nouvelle tâche</h3>
 <div class="fr"><div class="fg"><label>Libellé</label><input id="mt-l"></div><div class="fg"><label>Activité</label><select id="mt-a"></select></div></div>
 <div class="fr"><div class="fg"><label>Date début</label><input type="date" id="mt-d"></div><div class="fg"><label>Date fin</label><input type="date" id="mt-f"></div></div>
 <div class="fr"><div class="fg"><label>Priorité</label><select id="mt-p"><option>Normale</option><option>Haute</option><option>Urgente</option></select></div><div class="fg"><label>Statut</label><select id="mt-s"><option>Non demarre</option><option>En cours</option></select></div></div>
 <div class="fg"><label>Description</label><textarea id="mt-desc" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-tache')">Annuler</button><button class="btn bbr" onclick="saveTache()">Enregistrer</button></div>
</div></div>

<!-- AFFECTER -->
<div class="mo" id="mo-affect"><div class="mobox">
 <h3>👤 Affecter un subordonné</h3>
 <input type="hidden" id="aff-tid">
 <div class="fg"><label>Tâche</label><input id="aff-tl" readonly style="background:var(--cr)"></div>
 <div class="fg"><label>Subordonné direct</label><select id="aff-pid"></select></div>
 <div class="fg"><label>Rôle dans la tâche</label><input id="aff-r" value="Executant"></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-affect')">Annuler</button><button class="btn bbr" onclick="saveAffect()">✅ Affecter & Notifier</button></div>
</div></div>

<!-- ÉVALUER -->
<div class="mo" id="mo-noter"><div class="mobox">
 <h3>📊 Évaluation de performance</h3>
 <input type="hidden" id="n-pid">
 <div class="fg"><label>Collaborateur</label><input id="n-nom" readonly style="background:var(--cr)"></div>
 <div class="fr"><div class="fg"><label>Efficacité (%)</label><input type="number" id="n-eff" min="0" max="100"></div><div class="fg"><label>Note /20</label><input type="number" id="n-note" min="0" max="20"></div></div>
 <div class="fg"><label>Prime (optionnel)</label><input id="n-prime" placeholder="ex: 75 000 FCFA"></div>
 <div class="fg"><label>Commentaire</label><textarea id="n-comm" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-noter')">Annuler</button><button class="btn bok" onclick="saveNote()">Enregistrer</button></div>
</div></div>

<!-- EMPLOYÉ DU MOIS -->
<div class="mo" id="mo-em"><div class="mobox">
 <h3>🏆 Désigner l'Employé du mois</h3>
 <div class="fr"><div class="fg"><label>Mois</label><select id="em-mois"><option value="1">Janvier</option><option value="2">Février</option><option value="3">Mars</option><option value="4">Avril</option><option value="5">Mai</option><option value="6">Juin</option><option value="7">Juillet</option><option value="8">Août</option><option value="9">Septembre</option><option value="10">Octobre</option><option value="11">Novembre</option><option value="12">Décembre</option></select></div><div class="fg"><label>Année</label><input type="number" id="em-annee" value="2025"></div></div>
 <div class="fg"><label>Agent désigné</label><select id="em-pid"></select></div>
 <div class="fg"><label>Note finale /20</label><input type="number" id="em-note" min="0" max="20" step="0.1"></div>
 <div class="fg"><label>Motif de la désignation</label><textarea id="em-motif" rows="3" placeholder="Décrivez les mérites de cet agent..."></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-em')">Annuler</button><button class="btn bgd" onclick="saveEM()">🏆 Désigner</button></div>
</div></div>

<!-- TÉLÉPAIEMENT -->
<div class="mo" id="mo-tp"><div class="mobox" style="max-width:560px">
 <h3>💳 Nouveau télépaiement</h3>
 <div class="fr"><div class="fg"><label>Contribuable</label><input id="tp-cn"></div><div class="fg"><label>NIF / NINEA</label><input id="tp-nif" placeholder="SN000000"></div></div>
 <div class="fr"><div class="fg"><label>Type d'impôt</label><select id="tp-type"><option>Impôt sur les Sociétés</option><option>TVA mensuelle</option><option>Taxe foncière</option><option>Acompte IS</option><option>Taxe sur les salaires</option><option>Droits d'enregistrement</option><option>Taxe de plus-value immobilière</option><option>Droits de bornage</option><option>Contribution foncière bâtie</option><option>Autre</option></select></div><div class="fg"><label>Bureau</label><select id="tp-b"></select></div></div>
 <div class="fr"><div class="fg"><label>Montant dû (FCFA)</label><input type="number" id="tp-mt"></div><div class="fg"><label>Échéance</label><input type="date" id="tp-ech"></div></div>
 <div class="fg"><label>Notes</label><textarea id="tp-notes" rows="2"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-tp')">Annuler</button><button class="btn bbr" onclick="saveTP()">Enregistrer</button></div>
</div></div>

<!-- TP UPDATE -->
<div class="mo" id="mo-tpu"><div class="mobox">
 <h3>✏️ Mise à jour du paiement</h3>
 <input type="hidden" id="tpu-id">
 <div class="fg"><label>Statut</label><select id="tpu-statut"><option>En attente</option><option>Partiel</option><option>Paye</option><option>En retard</option></select></div>
 <div class="fr"><div class="fg"><label>Montant reçu (FCFA)</label><input type="number" id="tpu-mt"></div><div class="fg"><label>Mode de paiement</label><select id="tpu-mode"><option>Virement bancaire</option><option>Mobile Money (Wave)</option><option>Mobile Money (Orange)</option><option>Carte bancaire</option><option>Espèces (guichet)</option><option>Chèque</option></select></div></div>
 <div class="fg"><label>Notes</label><textarea id="tpu-notes" rows="2"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-tpu')">Annuler</button><button class="btn bok" onclick="saveTPU()">Confirmer</button></div>
</div></div>

<!-- PERSONNEL -->
<div class="mo" id="mo-personnel"><div class="mobox" style="max-width:560px">
 <h3>👥 Ajouter un membre du personnel</h3>
 <div style="background:var(--gdp);border-radius:9px;padding:10px 14px;margin-bottom:14px;font-size:.78rem;color:var(--br)">
  ℹ️ L'email et le mot de passe seront générés automatiquement et affichés après l'enregistrement.
 </div>
 <div class="fr"><div class="fg"><label>Nom</label><input id="mp-n"></div><div class="fg"><label>Prénom</label><input id="mp-pr"></div></div>
 <div class="fr"><div class="fg"><label>Téléphone</label><input id="mp-t"></div><div class="fg"><label>Date de naissance</label><input type="date" id="mp-dn"></div></div>
 <div class="fr"><div class="fg"><label>Rôle</label><select id="mp-r"><option value="chef_bureau">Chef de Bureau</option><option value="controleur">Contrôleur</option><option value="agent">Agent</option></select></div><div class="fg"><label>Bureau</label><select id="mp-b"></select></div></div>
 <div class="fr"><div class="fg"><label>Supérieur N+1</label><select id="mp-s"><option value="">-- Aucun --</option></select></div><div class="fg"><label>Fonction</label><input id="mp-f"></div></div>
 <div class="fg"><label>Photo (optionnel)</label>
  <input type="file" id="mp-photo-file" accept="image/*" onchange="previewMPPhoto(this)" style="padding:6px">
  <div id="mp-photo-preview" style="margin-top:8px;display:none"><img id="mp-photo-img" style="width:60px;height:60px;border-radius:50%;object-fit:cover;border:2px solid var(--gd)"></div>
  <input type="hidden" id="mp-photo-data">
 </div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-personnel')">Annuler</button><button class="btn bbr" onclick="savePersonnel()">Enregistrer</button></div>
</div></div>

<!-- CREDENTIALS (après ajout personnel) -->
<div class="mo" id="mo-cred"><div class="mobox">
 <h3>✅ Personnel ajouté avec succès</h3>
 <p style="font-size:.83rem;color:var(--mt);margin-bottom:12px">Transmettez ces identifiants au nouvel employé. Il peut se connecter immédiatement.</p>
 <div class="cred-box">
  <div class="c-row"><b>📧 Email</b><span id="cred-email"></span></div>
  <div class="sep"></div>
  <div class="c-row"><b>🔑 Mot de passe</b><span id="cred-pwd"></span></div>
 </div>
 <div style="background:#FDE8E8;border-radius:8px;padding:9px 13px;font-size:.76rem;color:var(--er)">
  ⚠️ Ces identifiants ne s'affichent qu'une seule fois. Notez-les avant de fermer.
 </div>
 <div class="moftr"><button class="btn bgd" onclick="cm('mo-cred');loadPersonnel()">Compris — Fermer</button></div>
</div></div>

<!-- STATUT EMPLOI -->
<div class="mo" id="mo-statut"><div class="mobox">
 <h3>⚙️ Modifier le statut de l'employé</h3>
 <input type="hidden" id="st-pid">
 <div class="fg"><label>Employé</label><input id="st-nom" readonly style="background:var(--cr)"></div>
 <div class="fg"><label>Nouveau statut</label>
  <select id="st-val">
   <option value="Actif">✅ Actif — Remet l'employé en activité</option>
   <option value="Réaffecté">🔄 Réaffecté — Nouvelles missions attribuées</option>
   <option value="Mis en pied">⛔ Mis en pied — Suspension temporaire</option>
  </select>
 </div>
 <div class="fg"><label>Commentaire (optionnel)</label><textarea id="st-comm" rows="2" placeholder="Raison ou instructions..."></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-statut')">Annuler</button><button class="btn bbr" onclick="saveStatut()">Confirmer</button></div>
</div></div>

<!-- PROPOSER -->
<div class="mo" id="mo-prop"><div class="mobox">
 <h3>💡 Proposer une activité à mon N+1</h3>
 <div class="fg"><label>Type d'activité</label><select id="pp-t"><option>Mission</option><option>Atelier</option><option>Séminaire</option><option>Forum</option><option>Formation</option></select></div>
 <div class="fg"><label>Description détaillée</label><textarea id="pp-d" rows="4" placeholder="Décrivez l'activité et son intérêt..."></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-prop')">Annuler</button><button class="btn bbr" onclick="saveProp()">Envoyer au N+1</button></div>
</div></div>

<!-- SIGNALEMENT -->
<div class="mo" id="mo-signal"><div class="mobox">
 <h3>⚠️ Faire un signalement</h3>
 <div class="fg"><label>Tâche concernée</label><select id="sg-t"><option value="">-- Aucune --</option></select></div>
 <div class="fg"><label>Description du problème</label><textarea id="sg-d" rows="4"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-signal')">Annuler</button><button class="btn bwn" onclick="saveSignal()">Envoyer au N+1</button></div>
</div></div>

<!-- COMPTE-RENDU -->
<div class="mo" id="mo-cr"><div class="mobox">
 <h3>📄 Soumettre un compte-rendu</h3>
 <div class="fg"><label>Tâche concernée</label><select id="cr-t"><option value="">-- Général --</option></select></div>
 <div class="fg"><label>Contenu</label><textarea id="cr-c" rows="6"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-cr')">Annuler</button><button class="btn bbr" onclick="saveCR()">Soumettre</button></div>
</div></div>

<!-- AVIS -->
<div class="mo" id="mo-avis"><div class="mobox">
 <h3>⭐ Donner un avis sur l'application</h3>
 <div class="fg"><label>Note globale</label><select id="av-n"><option value="5">⭐⭐⭐⭐⭐ Excellent</option><option value="4">⭐⭐⭐⭐ Bien</option><option value="3">⭐⭐⭐ Moyen</option><option value="2">⭐⭐ Insuffisant</option><option value="1">⭐ Mauvais</option></select></div>
 <div class="fg"><label>Commentaire</label><textarea id="av-c" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-avis')">Annuler</button><button class="btn bgd" onclick="saveAvis()">Soumettre</button></div>
</div></div>

<!-- GANTT -->
<div class="mo" id="mo-gantt"><div class="mobox" style="max-width:700px">
 <h3 id="gantt-t">Diagramme de Gantt</h3>
 <div id="gantt-c" style="margin-top:14px"></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-gantt')">Fermer</button></div>
</div></div>

<!-- HISTORIQUE AFFECTATIONS -->
<div class="mo" id="mo-hist"><div class="mobox">
 <h3>📜 Historique des affectations</h3>
 <div id="hist-c" style="margin-top:12px"></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-hist')">Fermer</button></div>
</div></div>

<!-- RESET CONFIRMATION -->
<div class="mo" id="mo-reset"><div class="mobox">
 <h3 style="color:var(--er)">⚠️ Réinitialisation complète</h3>
 <div style="background:#FDE8E8;border-radius:10px;padding:14px;margin-bottom:16px;font-size:.85rem;color:var(--er);line-height:1.6">
  <b>Cette action est irréversible.</b><br>
  Toutes les données seront supprimées :<br>
  tâches, activités, personnel, télépaiements, etc.<br>
  Un nouveau compte administrateur sera créé.
 </div>
 <div class="fg"><label>Tapez <b>CONFIRMER</b> pour valider</label><input id="reset-confirm" placeholder="CONFIRMER"></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-reset')">Annuler</button><button class="btn brd" style="padding:7px 14px;font-size:.8rem" onclick="doReset()">⚠️ Réinitialiser l'application</button></div>
</div></div>

<!-- RESET SUCCESS -->
<div class="mo" id="mo-reset-ok"><div class="mobox">
 <h3>✅ Application réinitialisée</h3>
 <p style="font-size:.83rem;color:var(--mt);margin-bottom:12px">L'application a été remise à zéro. Utilisez ces identifiants pour vous reconnecter :</p>
 <div class="cred-box">
  <div class="c-row"><b>📧 Email</b><span id="ro-email"></span></div>
  <div class="sep"></div>
  <div class="c-row"><b>🔑 Mot de passe</b><span id="ro-pwd"></span></div>
 </div>
 <div class="moftr"><button class="btn bgd" onclick="location.reload()">Se reconnecter</button></div>
</div></div>

<div id="toast"></div>
"""

HTML_JS = r"""
<script>
let CU=null, charts={};
const RFR={chef_centre:'Chef de Centre',chef_bureau:'Chef de Bureau',controleur:'Contrôleur',agent:'Agent'};
const RCOL={chef_centre:'#5C2E0E',chef_bureau:'#145c3a',controleur:'#7a4a08',agent:'#3d3d3d'};
const MFR=['','Janvier','Février','Mars','Avril','Mai','Juin','Juillet','Août','Septembre','Octobre','Novembre','Décembre'];

function api(u,o={}){return fetch(u,{credentials:'include',headers:{'Content-Type':'application/json'},...o}).then(r=>r.json())}
function post(u,d){return api(u,{method:'POST',body:JSON.stringify(d)})}
function put(u,d){return api(u,{method:'PUT',body:JSON.stringify(d)})}
function del(u){return api(u,{method:'DELETE'})}
function om(id){document.getElementById(id).classList.add('open')}
function cm(id){document.getElementById(id).classList.remove('open')}
document.querySelectorAll('.mo').forEach(m=>m.addEventListener('click',e=>{if(e.target===m)m.classList.remove('open')}));

function toast(msg,type='info'){
 const t=document.getElementById('toast');
 t.textContent=msg;t.style.display='block';
 t.style.background=type==='err'?'#8B1919':type==='ok'?'#2E7D52':type==='wn'?'#C17B1A':'#5C2E0E';
 try{
  const ac=new AudioContext(),osc=ac.createOscillator(),g=ac.createGain();
  osc.connect(g);g.connect(ac.destination);
  osc.frequency.setValueAtTime(type==='err'?220:type==='ok'?880:660,ac.currentTime);
  osc.frequency.exponentialRampToValueAtTime(type==='err'?180:type==='ok'?1100:800,ac.currentTime+0.1);
  g.gain.setValueAtTime(0.1,ac.currentTime);g.gain.exponentialRampToValueAtTime(0.001,ac.currentTime+0.25);
  osc.start();osc.stop(ac.currentTime+0.25);
 }catch(e){}
 setTimeout(()=>t.style.display='none',3500);
}

function badge(s){
 const m={Terminee:'bs',Paye:'bs','En cours':'bw',Partiel:'bw',Planifiee:'bi','Non demarre':'bm',
  'Non lue':'bw',Lue:'bm',Ouvert:'be',Traite:'bs',Acceptee:'bs',Refusee:'be','En attente':'bw',
  Soumise:'bi',Soumis:'bw',Valide:'bs',Brouillon:'bm',Haute:'be',Normale:'bi',Urgente:'bp2',
  'En retard':'be',Actif:'bs','Mis en pied':'be','Réaffecté':'bw'};
 return `<span class="badge ${m[s]||'bm'}">${s||'—'}</span>`;
}
function stars(n){return'<span style="color:var(--gd)">'+('★'.repeat(n)+'☆'.repeat(5-n))+'</span>';}
function fmt(n){return n!=null?Number(n).toLocaleString('fr-FR'):'—';}
function pct(a,b){return b?Math.round(a/b*100):0;}
function av(nom,prenom,role,photo){
 if(photo) return `<img src="${photo}" style="width:36px;height:36px;border-radius:50%;object-fit:cover;border:2px solid var(--gd);flex-shrink:0">`;
 const i=((nom||'?')[0]+(prenom||'?')[0]).toUpperCase();
 return `<div class="avatar" style="background:${RCOL[role]||'#888'};color:#fff">${i}</div>`;
}

function showPage(name){
 document.querySelectorAll('.page').forEach(p=>p.classList.remove('active'));
 document.querySelectorAll('.ni').forEach(n=>n.classList.remove('active'));
 document.getElementById('pg-'+name)?.classList.add('active');
 document.querySelector(`[data-p="${name}"]`)?.classList.add('active');
 LOADS[name]?.();
}

async function doLogin(){
 const err=document.getElementById('lerr');
 err.style.display='none';
 const r=await post('/api/login',{email:document.getElementById('l-e').value,password:document.getElementById('l-p').value});
 if(r.error){err.textContent=r.error;err.style.display='block';return;}
 CU=r;initApp();
}

async function doLogout(){
 await post('/api/logout',{});CU=null;
 document.getElementById('app').style.display='none';
 document.getElementById('login-screen').style.display='flex';
}

function initApp(){
 document.getElementById('login-screen').style.display='none';
 document.getElementById('app').style.display='flex';
 document.getElementById('rbadge').textContent=RFR[CU.role]||CU.role;
 document.getElementById('uname').textContent=`${CU.prenom||''} ${CU.nom}`;
 buildNav();loadNotifs();populateSels();
 showPage('dashboard');
 setInterval(loadNotifs,30000);
}

function buildNav(){
 const r=CU.role;
 const items=[
  {s:'Tableau de bord'},{p:'dashboard',i:'⊞',l:'Vue générale'},
  {s:'Activités & Tâches'},
  {p:'activites',i:'📋',l:'Activités'},{p:'taches',i:'✅',l:'Tâches'},
  ...(r!=='agent'?[{p:'affecter',i:'👤',l:'Affectations'}]:[]),
  {s:'Mon équipe'},
  ...(r!=='agent'?[{p:'personnel',i:'👥',l:'Personnel'}]:[]),
  {p:'performances',i:'📊',l:r==='agent'?'Mes évaluations':'Performances'},
  {p:'employe-mois',i:'🏆',l:'Employé du mois'},
  {s:'Fiscal & Paiement'},
  {p:'telepaiement',i:'💳',l:'Télépaiements'},
  {s:'Communication'},
  {p:'signalements',i:'⚠️',l:'Signalements'},
  {p:'propositions',i:'💡',l:['chef_bureau','chef_centre'].includes(r)?'Propositions reçues':'Proposer activité'},
  ...(['chef_bureau','controleur','agent'].includes(r)?[{p:'cr',i:'📄',l:'Comptes-rendus'}]:[]),
  {p:'avis',i:'⭐',l:'Avis application'},
  ...(r==='chef_centre'?[{s:'Administration'},{p:'bureaux',i:'🏢',l:'Bureaux'}]:[]),
 ];
 document.getElementById('nav').innerHTML=items.map(i=>
  i.s?`<div class="nsec">${i.s}</div>`:
  `<div class="ni" data-p="${i.p}" onclick="showPage('${i.p}')"><span class="nico">${i.i}</span>${i.l}</div>`
 ).join('');
}

/* ── NOTIFICATIONS ── */
async function loadNotifs(){
 const rows=await api('/api/notifications');if(rows.error)return;
 const unread=rows.filter(r=>r.statut==='Non lue');
 const cnt=document.getElementById('nc');
 cnt.textContent=unread.length;cnt.style.display=unread.length?'flex':'none';
 if(window._nc!==undefined&&unread.length>window._nc&&unread.length>0)toast(`🔔 ${unread.length} notification(s) non lue(s)`);
 window._nc=unread.length;
 const tico={tache:'✅',performance:'📊',employe_mois:'🏆',proposition:'💡',alerte:'⚠️',info:'ℹ️'};
 document.getElementById('nlist').innerHTML=rows.length===0
  ?'<div class="empty"><div class="eico">🔔</div><p>Aucune notification</p></div>'
  :rows.map(r=>`<div class="nitem ${r.statut==='Non lue'?'unread':''}" onclick="lireNotif(${r.notification_id},this)">
   <div class="ntit">${tico[r.type_notif]||'•'} ${r.titre}</div>
   <div class="ndesc">${r.description}</div>
   <div class="ntime">${(r.dateEnvoie||'').substring(0,16)}</div></div>`).join('');
}
async function lireNotif(id,el){await put('/api/notifications/'+id+'/lire',{});el.classList.remove('unread');loadNotifs();}
async function toutLire(){await put('/api/notifications/tout-lire',{});loadNotifs();toast('Toutes les notifications lues','ok');}
function toggleNP(){document.getElementById('np').classList.toggle('open');}

function mkChart(id,data,type){
 if(charts[id])charts[id].destroy();
 const el=document.getElementById(id);if(!el)return;
 const lk=Object.keys(data[0]||{}).find(k=>!['n','total','total_m','paye_m'].includes(k));
 const vk=data[0]&&('total_m' in data[0])?'total_m':'n';
 const C=['#5C2E0E','#C9A840','#2E7D52','#B71C1C','#1565C0','#C17B1A','#8B5A3A','#4A148C'];
 charts[id]=new Chart(el,{type,data:{labels:data.map(r=>r[lk]),datasets:[{data:data.map(r=>r[vk]),backgroundColor:C,borderColor:type==='bar'?C:'#fff',borderWidth:2,borderRadius:type==='bar'?5:0}]},
  options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{position:type==='doughnut'?'right':'bottom',labels:{font:{size:11},color:'#2D1A0E'}}}}});
}

/* ── DASHBOARD ── */
async function loadDashboard(){
 const d=await api('/api/dashboard');if(d.error)return;
 const r=CU.role;const el=document.getElementById('pg-dashboard');
 if(r==='chef_centre'){
  const tpPaye=d.tp_stats?.find(r=>r.statut==='Paye')?.total||0;
  el.innerHTML=`<div class="ptitle"><div class="ptitle-ico">⊞</div>Vue d'ensemble — Chef de Centre
   <button class="btn brd bsm" onclick="om('mo-reset')" style="margin-left:auto">🔄 Réinitialiser</button></div>
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_bureaux}</div><div class="l">Bureaux</div></div>
    <div class="stat gd"><div class="n">${d.nb_personnel}</div><div class="l">Personnel actif</div></div>
    <div class="stat"><div class="n">${d.nb_activites}</div><div class="l">Activités</div></div>
    <div class="stat wn"><div class="n">${d.nb_taches}</div><div class="l">Tâches</div></div>
    <div class="stat ok"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    ${d.nb_suspendus>0?`<div class="stat er"><div class="n">${d.nb_suspendus}</div><div class="l">Mis en pied</div></div>`:''}
   </div>
   ${d.employe_mois?.length?`<div class="g3" style="margin-bottom:18px">${d.employe_mois.map(em=>`<div class="em-card"><div class="em-month">${MFR[em.mois]} ${em.annee}</div><div class="em-name">${em.nom} ${em.prenom||''}</div><div class="em-bureau">${em.bureau_nom||''}</div><div class="em-note">Note : ${em.note_finale}/20</div></div>`).join('')}</div>`:''}
   <div class="g3">
    <div class="card"><div class="chead"><h3>Activités / statut</h3></div><div class="cbody"><div class="cht"><canvas id="ch-a"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Tâches / statut</h3></div><div class="cbody"><div class="cht"><canvas id="ch-t"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Activités / bureau</h3></div><div class="cbody"><div class="cht"><canvas id="ch-b"></canvas></div></div></div>
   </div>
   <div class="card" style="margin-top:16px"><div class="chead"><h3>🏅 Dernières performances</h3></div><div class="cbody">
    <table><thead><tr><th>Agent</th><th>Bureau</th><th>Note</th><th>Efficacité</th></tr></thead><tbody>
    ${(d.top_performances||[]).map(p=>`<tr><td><b>${p.nom} ${p.prenom||''}</b></td><td>${p.bureau_nom||'—'}</td><td><b style="font-size:1rem;color:${p.note>=15?'#2E7D52':p.note>=10?'#C17B1A':'#B71C1C'}">${p.note}</b>/20</td><td><div class="pgbar"><div class="pgfill" style="width:${p.efficacite}%"></div></div></td></tr>`).join('')}
    </tbody></table></div></div>`;
  mkChart('ch-a',d.activites_statut,'doughnut');
  mkChart('ch-t',d.taches_statut,'doughnut');
  mkChart('ch-b',d.bureaux_activites,'bar');
 } else if(r==='chef_bureau'){
  el.innerHTML=`<div class="ptitle"><div class="ptitle-ico">⊞</div>Mon Bureau — Chef de Bureau</div>
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_controleurs}</div><div class="l">Contrôleurs</div></div>
    <div class="stat gd"><div class="n">${d.nb_agents}</div><div class="l">Agents</div></div>
    <div class="stat"><div class="n">${d.nb_activites}</div><div class="l">Activités</div></div>
    <div class="stat wn"><div class="n">${d.nb_taches}</div><div class="l">Tâches</div></div>
    <div class="stat ok"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    <div class="stat er"><div class="n">${d.signalements_ouverts}</div><div class="l">Signalements</div></div>
   </div>
   ${d.employe_mois?.length?`<div style="margin-bottom:18px">${d.employe_mois.map(em=>`<div class="em-card" style="max-width:360px"><div class="em-month">Employé du mois — ${MFR[em.mois]} ${em.annee}</div><div class="em-name">${em.nom} ${em.prenom||''}</div><div class="em-note">${em.motif||''}</div></div>`).join('')}</div>`:''}
   <div class="g2">
    <div class="card"><div class="chead"><h3>Tâches du bureau</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tb"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Télépaiements du bureau</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tpb"></canvas></div></div></div>
   </div>`;
  mkChart('ch-tb',d.taches_statut,'doughnut');
  if(d.tp_bureau?.length)mkChart('ch-tpb',d.tp_bureau,'doughnut');
 } else if(r==='controleur'){
  el.innerHTML=`<div class="ptitle"><div class="ptitle-ico">⊞</div>Mon équipe — Contrôleur</div>
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_agents}</div><div class="l">Agents supervisés</div></div>
    <div class="stat gd"><div class="n">${d.nb_taches_assignees}</div><div class="l">Tâches assignées</div></div>
    <div class="stat wn"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    <div class="stat er"><div class="n">${d.signalements_recus}</div><div class="l">Signalements reçus</div></div>
   </div>
   <div class="g2">
    <div class="card"><div class="chead"><h3>Avancement des tâches</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tc"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Mes tâches</h3><button class="btn bo bsm" onclick="showPage('taches')">Tout voir</button></div><div class="cbody">
    ${(d.mes_taches||[]).slice(0,6).map(t=>`<div style="display:flex;justify-content:space-between;align-items:center;padding:8px 0;border-bottom:1px solid var(--cr2)">
     <div><div style="font-size:.82rem;font-weight:700">${t.libelle}</div><div style="font-size:.72rem;color:var(--mt)">${t.act_nom} · ${t.dateFin}</div></div>${badge(t.statut)}</div>`).join('')||'<div class="empty"><p>Aucune tâche</p></div>'}
    </div></div></div>`;
  mkChart('ch-tc',d.taches_statut,'doughnut');
 } else {
  const p=d.ma_performance;const curr=d.employe_mois_courant;
  el.innerHTML=`<div class="ptitle"><div class="ptitle-ico">⊞</div>Mon espace — Agent</div>
   ${curr?`<div class="em-card" style="margin-bottom:18px"><div class="em-month">🏆 Employé du mois — ${MFR[curr.mois]} ${curr.annee}</div><div class="em-name">${curr.nom} ${curr.prenom||''}</div></div>`:''}
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_taches}</div><div class="l">Tâches assignées</div></div>
    <div class="stat ok"><div class="n">${d.nb_terminees}</div><div class="l">Terminées</div></div>
    <div class="stat wn"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    ${p?`<div class="stat gd"><div class="n">${p.note}/20</div><div class="l">Dernière note</div></div>`:''}
   </div>
   <div class="card"><div class="chead"><h3>Mes tâches en cours</h3></div><div class="cbody">
   ${(d.mes_taches||[]).map(t=>`<div class="${t.priorite==='Haute'?'prio-H':t.priorite==='Urgente'?'prio-U':''}" style="display:flex;justify-content:space-between;align-items:center;padding:11px 8px;border-bottom:1px solid var(--cr2)">
    <div><div style="font-size:.86rem;font-weight:700">${t.libelle}</div><div style="font-size:.74rem;color:var(--mt)">${t.act_nom} · Échéance : <b>${t.dateFin}</b></div></div>
    <div class="acts">${badge(t.statut)}<button class="btn bo bsm" onclick="majStatut(${t.tache_id})">✏️</button><button class="btn bo bsm" onclick="openCR(${t.tache_id})">📄</button></div></div>`).join('')||'<div class="empty"><p>Aucune tâche assignée</p></div>'}
   </div></div>`;
 }
}

/* ── BUREAUX ── */
async function loadBureaux(){
 const rows=await api('/api/bureaux');
 document.getElementById('pg-bureaux').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">🏢</div>Bureaux <button class="btn bgd bsm" onclick="addBureau()">+ Ajouter</button></div>
  <div class="card"><table><thead><tr><th>Code</th><th>Nom</th><th>Description</th><th>Personnel</th><th>Activités</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td><b>${r.code||'—'}</b></td><td><b>${r.nom}</b></td><td style="font-size:.78rem;max-width:220px">${r.description||'—'}</td><td>${r.nb_pers}</td><td>${r.nb_act}</td></tr>`).join('')}
  </tbody></table></div>`;
}
async function addBureau(){
 const n=prompt('Nom du bureau :');if(!n)return;
 const c=prompt('Code (ex: GCS) :');
 await post('/api/bureaux',{nom:n,code:c});loadBureaux();populateSels();toast('Bureau ajouté','ok');
}

/* ── ACTIVITÉS ── */
async function loadActivites(){
 const rows=await api('/api/activites');
 const ce=['chef_centre','chef_bureau'].includes(CU.role);
 document.getElementById('pg-activites').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">📋</div>Activités ${ce?`<button class="btn bgd bsm" onclick="om('mo-activite')">+ Nouvelle</button>`:''}
  </div>
  <div class="card"><table><thead><tr><th>Nom</th><th>Type</th><th>Bureau</th><th>Début</th><th>Fin</th><th>Statut</th><th>Avancement</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td><b>${r.nom}</b></td><td>${r.type_activite||'—'}</td><td>${r.bureau_nom||'—'}</td>
   <td>${r.dateDebut||'—'}</td><td>${r.dateFin||'—'}</td><td>${badge(r.statut)}</td>
   <td><div style="font-size:.74rem;color:var(--mt)">${r.nb_ok||0}/${r.nb_taches||0} tâches</div><div class="pgbar"><div class="pgfill" style="width:${r.nb_taches?pct(r.nb_ok,r.nb_taches):0}%"></div></div></td>
   <td class="acts">
    <button class="btn bo bsm" onclick="showGantt(${r.activite_id},'${r.nom.replace(/'/g,"\\'")}')">📊 Gantt</button>
    ${ce?`<button class="btn brd" onclick="delAct(${r.activite_id})">🗑</button>`:''}
   </td></tr>`).join('')||'<tr><td colspan="8"><div class="empty"><div class="eico">📋</div><p>Aucune activité</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveActivite(){
 const r=await post('/api/activites',{nom:document.getElementById('ma-n').value,type_activite:document.getElementById('ma-t').value,description:document.getElementById('ma-desc').value,dateDebut:document.getElementById('ma-d').value,dateFin:document.getElementById('ma-f').value,bureau_id:document.getElementById('ma-b').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-activite');loadActivites();populateSels();toast('✅ Activité créée','ok');
}
async function delAct(id){if(!confirm('Supprimer cette activité ?'))return;await del('/api/activites/'+id);loadActivites();}
async function showGantt(aid,nom){
 const d=await api('/api/activites/'+aid+'/gantt');
 document.getElementById('gantt-t').textContent='📊 Gantt — '+nom;
 if(!d.taches?.length){document.getElementById('gantt-c').innerHTML='<div class="empty"><p>Aucune tâche</p></div>';om('mo-gantt');return;}
 const dates=d.taches.flatMap(t=>[t.dateDebut,t.dateFin]).filter(Boolean).sort();
 const s0=new Date(dates[0]),s1=new Date(dates[dates.length-1]);
 const tot=Math.max(1,(s1-s0)/86400000);
 const sc={Terminee:'#2E7D52','En cours':'#C9A840','Non demarre':'#5C2E0E','En retard':'#B71C1C'};
 let h=`<div style="font-size:.72rem;color:var(--mt);display:flex;justify-content:space-between;margin-bottom:8px"><span>${dates[0]}</span><span>${dates[dates.length-1]}</span></div>`;
 d.taches.forEach(t=>{
  const ts=new Date(t.dateDebut),tf=new Date(t.dateFin);
  const l=Math.max(0,(ts-s0)/86400000/tot*100);
  const w=Math.max(2,(tf-ts)/86400000/tot*100);
  h+=`<div style="display:flex;align-items:center;gap:8px;margin-bottom:8px">
   <div style="min-width:130px;font-size:.76rem;overflow:hidden;text-overflow:ellipsis;white-space:nowrap" title="${t.libelle}">${t.libelle}</div>
   <div style="flex:1;background:var(--cr2);border-radius:5px;height:20px;position:relative">
    <div style="position:absolute;left:${l}%;width:${w}%;height:100%;border-radius:5px;background:${sc[t.statut]||'#888'}" title="${t.statut}|${t.agents||''}"></div></div>
   ${badge(t.statut)}</div>`;
 });
 document.getElementById('gantt-c').innerHTML=h;om('mo-gantt');
}

/* ── TÂCHES ── */
async function loadTaches(){
 const rows=await api('/api/taches');
 const cm2=['chef_bureau','controleur'].includes(CU.role);
 const isAg=CU.role==='agent';
 document.getElementById('pg-taches').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">✅</div>Tâches ${cm2?`<button class="btn bgd bsm" onclick="om('mo-tache')">+ Nouvelle</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Libellé</th><th>Activité</th><th>Début</th><th>Échéance</th><th>Priorité</th><th>Statut</th><th>Agents</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr class="${r.priorite==='Haute'?'prio-H':r.priorite==='Urgente'?'prio-U':''}"><td><b>${r.libelle}</b></td>
   <td style="font-size:.78rem">${r.act_nom||'—'}</td><td>${r.dateDebut||'—'}</td><td><b>${r.dateFin||'—'}</b></td>
   <td>${badge(r.priorite)}</td><td>${badge(r.statut)}</td>
   <td style="font-size:.76rem;max-width:120px">${r.agents||'—'}</td>
   <td class="acts">
    ${cm2?`<button class="btn bo bsm" onclick="openAff(${r.tache_id},'${r.libelle.replace(/'/g,"\\'")}')">👤</button><button class="btn bo bsm" onclick="showHist(${r.tache_id})">📜</button><button class="btn brd" onclick="delTache(${r.tache_id})">🗑</button>`:''}
    ${isAg?`<button class="btn bo bsm" onclick="majStatut(${r.tache_id})">✏️ Statut</button><button class="btn bo bsm" onclick="openCR(${r.tache_id})">📄 CR</button>`:''}
   </td></tr>`).join('')||'<tr><td colspan="8"><div class="empty"><div class="eico">✅</div><p>Aucune tâche</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveTache(){
 const r=await post('/api/taches',{libelle:document.getElementById('mt-l').value,description:document.getElementById('mt-desc').value,dateDebut:document.getElementById('mt-d').value,dateFin:document.getElementById('mt-f').value,priorite:document.getElementById('mt-p').value,statut:document.getElementById('mt-s').value,activite_id:document.getElementById('mt-a').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-tache');loadTaches();toast('✅ Tâche créée','ok');
}
async function delTache(id){if(!confirm('Supprimer ?'))return;await del('/api/taches/'+id);loadTaches();}
async function majStatut(tid){
 const s=prompt('Nouveau statut:\n- Non demarre\n- En cours\n- Terminee\n- En retard');if(!s)return;
 await put('/api/taches/'+tid,{statut:s});loadTaches();toast('Statut mis à jour','ok');
}

/* ── AFFECTATIONS ── */
async function loadAffecter(){
 const rows=await api('/api/taches');
 document.getElementById('pg-affecter').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">👤</div>Gestion des affectations</div>
  <div class="card"><table><thead><tr><th>Tâche</th><th>Activité</th><th>Échéance</th><th>Statut</th><th>Agents affectés</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td><b>${r.libelle}</b></td><td>${r.act_nom||'—'}</td><td>${r.dateFin||'—'}</td>
   <td>${badge(r.statut)}</td><td style="font-size:.76rem">${r.agents||'<em style="color:var(--mt)">Aucun</em>'}</td>
   <td class="acts"><button class="btn bbr bsm" onclick="openAff(${r.tache_id},'${r.libelle.replace(/'/g,"\\'")}')">+ Affecter</button><button class="btn bo bsm" onclick="showHist(${r.tache_id})">📜 Historique</button></td></tr>`).join('')}
  </tbody></table></div>`;
}
function openAff(tid,label){
 document.getElementById('aff-tid').value=tid;
 document.getElementById('aff-tl').value=label||'Tâche #'+tid;
 om('mo-affect');
}
async function saveAffect(){
 const r=await post('/api/taches/'+document.getElementById('aff-tid').value+'/affecter',{personnel_id:document.getElementById('aff-pid').value,role:document.getElementById('aff-r').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-affect');loadNotifs();toast('✅ Agent affecté & notifié','ok');
}
async function showHist(tid){
 const rows=await api('/api/taches/'+tid+'/historique');
 document.getElementById('hist-c').innerHTML=!rows.length?'<div class="empty"><p>Aucune affectation</p></div>':`<table><thead><tr><th>Agent</th><th>Rôle</th><th>Affecté le</th><th>Retiré le</th><th>Actif</th><th>Par</th></tr></thead><tbody>${rows.map(r=>`<tr><td>${r.agent_nom}</td><td>${r.role_affect}</td><td>${r.date_affectation||'—'}</td><td>${r.date_retrait||'—'}</td><td>${r.actif?'✅':'❌'}</td><td>${r.sup_nom||'—'}</td></tr>`).join('')}</tbody></table>`;
 om('mo-hist');
}

/* ── PERSONNEL ── */
async function loadPersonnel(){
 const rows=await api('/api/personnel');
 const ce=['chef_centre','chef_bureau'].includes(CU.role);
 const isCC=CU.role==='chef_centre';
 document.getElementById('pg-personnel').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">👥</div>Personnel ${ce?`<button class="btn bgd bsm" onclick="om('mo-personnel')">+ Ajouter</button>`:''}
  </div>
  <div class="card"><table><thead><tr><th>Nom</th><th>Rôle</th><th>Fonction</th><th>Bureau</th><th>Email</th><th>Statut</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr style="${r.actif===0?'opacity:.5':r.statut_emploi==='Mis en pied'?'background:#fff8f8':''}">
   <td><div style="display:flex;align-items:center;gap:9px">${av(r.nom,r.prenom,r.role,r.photo)}<div><b>${r.nom} ${r.prenom||''}</b><div style="font-size:.71rem;color:var(--mt)">${r.email}</div></div></div></td>
   <td><span class="rc rc-${r.role}">${RFR[r.role]||r.role}</span></td>
   <td>${r.fonction||'—'}</td><td>${r.bureau_nom||'—'}</td>
   <td style="font-size:.74rem">${r.email}</td>
   <td class="statut-${r.statut_emploi==='Actif'?'actif':r.statut_emploi==='Mis en pied'?'pied':'reaffect'}">${r.actif===0?'Désactivé':r.statut_emploi||'Actif'}</td>
   <td class="acts">
    ${(ce&&r.role!=='chef_centre')?`<button class="btn bo bsm" onclick="openStatut(${r.personnel_id},'${(r.nom+' '+(r.prenom||'')).trim().replace(/'/g,"\\'")}','${r.statut_emploi||'Actif'}')">⚙️</button>`:''}
    <button class="btn bok bsm" onclick="openNoter(${r.personnel_id},'${(r.nom+' '+(r.prenom||'')).trim().replace(/'/g,"\\'")}')">📊 Éval.</button>
    <button class="btn bgd bsm" onclick="openEM(${r.personnel_id})">🏆</button>
    ${(ce&&r.actif===0)?`<button class="btn bok bsm" onclick="reactiver(${r.personnel_id})">🔓 Réactiver</button>`:''}
    ${(ce&&r.actif===1&&r.role!=='chef_centre')?`<button class="btn brd" onclick="delPers(${r.personnel_id})">🗑</button>`:''}
   </td></tr>`).join('')||'<tr><td colspan="7"><div class="empty"><div class="eico">👥</div><p>Aucun personnel</p></div></td></tr>'}
  </tbody></table></div>`;
}
function previewMPPhoto(input){
 const file=input.files[0];if(!file)return;
 const reader=new FileReader();
 reader.onload=e=>{
  document.getElementById('mp-photo-img').src=e.target.result;
  document.getElementById('mp-photo-preview').style.display='block';
  document.getElementById('mp-photo-data').value=e.target.result;
 };
 reader.readAsDataURL(file);
}
async function savePersonnel(){
 const r=await post('/api/personnel',{
  nom:document.getElementById('mp-n').value,prenom:document.getElementById('mp-pr').value,
  telephone:document.getElementById('mp-t').value,dateNaissance:document.getElementById('mp-dn').value,
  fonction:document.getElementById('mp-f').value,role:document.getElementById('mp-r').value,
  bureau_id:document.getElementById('mp-b').value,superieur_id:document.getElementById('mp-s').value||null,
  photo:document.getElementById('mp-photo-data').value||null
 });
 if(r.error){toast(r.error,'err');return;}
 cm('mo-personnel');
 document.getElementById('cred-email').textContent=r.email;
 document.getElementById('cred-pwd').textContent=r.password;
 om('mo-cred');
 populateSels();toast('✅ Personnel ajouté','ok');
}
async function delPers(id){
 if(!confirm('Désactiver ce personnel ? Son accès sera révoqué.'))return;
 await del('/api/personnel/'+id);loadPersonnel();toast('Personnel désactivé','wn');
}
async function reactiver(id){
 await put('/api/personnel/'+id+'/reactiver',{});loadPersonnel();toast('Personnel réactivé','ok');
}
function openStatut(pid,nom,statut){
 document.getElementById('st-pid').value=pid;
 document.getElementById('st-nom').value=nom;
 document.getElementById('st-val').value=statut;
 document.getElementById('st-comm').value='';
 om('mo-statut');
}
async function saveStatut(){
 const pid=document.getElementById('st-pid').value;
 const r=await put('/api/personnel/'+pid+'/statut',{statut_emploi:document.getElementById('st-val').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-statut');loadPersonnel();toast('Statut mis à jour','ok');
}

/* ── PERFORMANCES ── */
async function loadPerformances(){
 const rows=await api('/api/performances');
 const ce=['chef_bureau','controleur'].includes(CU.role);
 document.getElementById('pg-performances').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">📊</div>Performances ${ce?`<button class="btn bgd bsm" onclick="openNoterModal()">+ Évaluer</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Agent</th><th>Note</th><th>Efficacité</th><th>Prime</th><th>Commentaire</th><th>Mois</th>${CU.role!=='agent'?'<th>Évaluateur</th>':''}</tr></thead><tbody>
  ${rows.map(r=>`<tr><td><b>${r.nom||'Moi'} ${r.prenom||''}</b></td>
   <td><b style="font-size:1rem;color:${r.note>=15?'#2E7D52':r.note>=10?'#C17B1A':'#B71C1C'}">${r.note}</b>/20</td>
   <td><div style="display:flex;align-items:center;gap:7px"><div class="pgbar" style="width:70px"><div class="pgfill" style="width:${r.efficacite||0}%;background:${r.efficacite>=75?'#2E7D52':r.efficacite>=50?'#C17B1A':'#B71C1C'}"></div></div>${r.efficacite||0}%</div></td>
   <td>${r.prime||'—'}</td><td style="max-width:200px;font-size:.78rem">${r.commentaire||'—'}</td>
   <td>${r.mois?MFR[r.mois]+' '+r.annee:r.date_eval||'—'}</td>
   ${CU.role!=='agent'?`<td>${r.eval_nom||'—'}</td>`:''}</tr>`).join('')||
   '<tr><td colspan="7"><div class="empty"><div class="eico">📊</div><p>Aucune évaluation</p></div></td></tr>'}
  </tbody></table></div>`;
}
function openNoterModal(){document.getElementById('n-pid').value='';document.getElementById('n-nom').value='';om('mo-noter');}
function openNoter(pid,nom){document.getElementById('n-pid').value=pid;document.getElementById('n-nom').value=nom;om('mo-noter');}
async function saveNote(){
 const pid=document.getElementById('n-pid').value;if(!pid){toast('Sélectionnez un agent','err');return;}
 const r=await post('/api/performances',{personnel_id:pid,efficacite:document.getElementById('n-eff').value,note:document.getElementById('n-note').value,prime:document.getElementById('n-prime').value,commentaire:document.getElementById('n-comm').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-noter');loadPerformances();loadNotifs();toast('✅ Évaluation enregistrée','ok');
}

/* ── EMPLOYÉ DU MOIS ── */
async function loadEmployeMois(){
 const rows=await api('/api/employe-mois');
 const ce=['chef_centre','chef_bureau'].includes(CU.role);
 document.getElementById('pg-employe-mois').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">🏆</div>Employé du mois ${ce?`<button class="btn bgd bsm" onclick="om('mo-em')">+ Désigner</button>`:''}</div>
  <div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:14px;margin-bottom:20px">
  ${rows.map(em=>`<div class="em-card"><div class="em-month">${MFR[em.mois]} ${em.annee}</div>
   <div class="em-name">${em.nom} ${em.prenom||''}</div>
   <div class="em-bureau">${em.bureau_nom||''} · ${em.fonction||''}</div>
   <div class="em-note" style="margin-top:8px"><b>Note : ${em.note_finale}/20</b></div>
   <div class="em-note" style="margin-top:4px;font-size:.76rem;opacity:.85">${em.motif||''}</div>
   <div class="em-note" style="margin-top:6px;font-size:.7rem;opacity:.7">Désigné par : ${em.designateur_nom||'—'}</div>
  </div>`).join('')||'<p style="color:var(--mt)">Aucune désignation pour l\'instant.</p>'}
  </div>`;
}
function openEM(pid){document.getElementById('em-pid').value=pid;document.getElementById('em-motif').value='';om('mo-em');}
async function saveEM(){
 const r=await post('/api/employe-mois',{personnel_id:document.getElementById('em-pid').value,mois:document.getElementById('em-mois').value,annee:document.getElementById('em-annee').value,note_finale:document.getElementById('em-note').value,motif:document.getElementById('em-motif').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-em');loadEmployeMois();loadNotifs();toast('🏆 Employé du mois désigné !','ok');
}

/* ── TÉLÉPAIEMENT ── */
async function loadTelepaiement(){
 const [rows,stats]=await Promise.all([api('/api/telepaiements'),api('/api/telepaiements/stats')]);
 const total=fmt(Math.round((stats.total_attendu||0)*1000000));
 const paye=fmt(Math.round((stats.total_recouvre||0)*1000000));
 const taux=stats.total_attendu?Math.round(stats.total_recouvre/stats.total_attendu*100):0;
 document.getElementById('pg-telepaiement').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">💳</div>Télépaiements <button class="btn bgd bsm" onclick="om('mo-tp')">+ Nouveau</button></div>
  <div class="gstats" style="margin-bottom:16px">
   <div class="stat"><div class="n">${rows.length}</div><div class="l">Dossiers</div></div>
   <div class="stat ok"><div class="n" style="font-size:1.2rem">${total} F</div><div class="l">Montant attendu</div></div>
   <div class="stat gd"><div class="n" style="font-size:1.2rem">${paye} F</div><div class="l">Montant recouvré</div></div>
   <div class="stat wn"><div class="n">${taux}%</div><div class="l">Taux recouvrement</div></div>
  </div>
  <div class="g2" style="margin-bottom:16px">
   <div class="card"><div class="chead"><h3>Par statut</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tp-s"></canvas></div></div></div>
   <div class="card"><div class="chead"><h3>Par type d'impôt (MFCFA)</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tp-t"></canvas></div></div></div>
  </div>
  <div class="card"><table><thead><tr><th>Référence</th><th>Contribuable</th><th>Type impôt</th><th>Montant dû</th><th>Reçu</th><th>Statut</th><th>Échéance</th><th>Bureau</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr>
   <td><b>${r.reference}</b></td><td>${r.contribuable_nom}</td>
   <td style="font-size:.78rem">${r.type_impot}</td>
   <td><b>${fmt(r.montant)}</b></td>
   <td style="color:${r.statut==='Paye'?'#2E7D52':r.statut==='En retard'?'#B71C1C':'#C17B1A'};font-weight:700">${fmt(r.montant_paye)}</td>
   <td>${badge(r.statut)}</td><td>${r.date_echeance||'—'}</td>
   <td style="font-size:.76rem">${r.bureau_nom||'—'}</td>
   <td><button class="btn bbr bsm" onclick="openTPU(${r.tp_id},${r.montant},'${r.statut}')">✏️</button></td>
  </tr>`).join('')||'<tr><td colspan="9"><div class="empty"><div class="eico">💳</div><p>Aucun dossier</p></div></td></tr>'}
  </tbody></table></div>`;
 if(stats.par_statut?.length)mkChart('ch-tp-s',stats.par_statut,'doughnut');
 if(stats.par_type?.length)mkChart('ch-tp-t',stats.par_type,'bar');
}
async function saveTP(){
 const r=await post('/api/telepaiements',{contribuable_nom:document.getElementById('tp-cn').value,contribuable_nif:document.getElementById('tp-nif').value,type_impot:document.getElementById('tp-type').value,montant:document.getElementById('tp-mt').value,bureau_id:document.getElementById('tp-b').value,date_echeance:document.getElementById('tp-ech').value,notes:document.getElementById('tp-notes').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-tp');loadTelepaiement();toast(`✅ Télépaiement ${r.reference} créé`,'ok');
}
function openTPU(id,montant,statut){
 document.getElementById('tpu-id').value=id;document.getElementById('tpu-mt').value=montant;
 document.getElementById('tpu-statut').value=statut==='En attente'?'Partiel':statut;
 om('mo-tpu');
}
async function saveTPU(){
 const id=document.getElementById('tpu-id').value;
 await put('/api/telepaiements/'+id,{statut:document.getElementById('tpu-statut').value,montant_paye:parseFloat(document.getElementById('tpu-mt').value),mode_paiement:document.getElementById('tpu-mode').value,notes:document.getElementById('tpu-notes').value});
 cm('mo-tpu');loadTelepaiement();toast('✅ Paiement mis à jour','ok');
}

/* ── SIGNALEMENTS ── */
async function loadSignalements(){
 const rows=await api('/api/signalements');
 const isN=['chef_centre','chef_bureau','controleur'].includes(CU.role);
 document.getElementById('pg-signalements').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">⚠️</div>Signalements <button class="btn bgd bsm" onclick="om('mo-signal')">+ Signaler</button></div>
  <div class="card"><table><thead><tr><th>Description</th>${isN?'<th>Auteur</th>':''}<th>Tâche</th><th>Date</th><th>Statut</th>${isN?'<th>Actions</th>':''}</tr></thead><tbody>
  ${rows.map(r=>`<tr><td style="max-width:220px">${r.description}</td>${isN?`<td>${r.auteur||'—'}</td>`:''}
   <td>${r.tache_lib||'—'}</td><td>${r.dateEnvoie||'—'}</td><td>${badge(r.statut)}</td>
   ${isN&&r.statut==='Ouvert'?`<td><button class="btn bok bsm" onclick="traiterSignal(${r.signalement_id})">✅ Traiter</button></td>`:(isN?'<td>—</td>':'')}</tr>`).join('')||
   '<tr><td colspan="6"><div class="empty"><div class="eico">⚠️</div><p>Aucun signalement</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveSignal(){
 const r=await post('/api/signalements',{description:document.getElementById('sg-d').value,tache_id:document.getElementById('sg-t').value||null});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-signal');loadSignalements();toast('✅ Signalement envoyé','ok');
}
async function traiterSignal(id){
 const rep=prompt('Réponse / action entreprise :');
 await put('/api/signalements/'+id+'/repondre',{statut:'Traite',reponse:rep});loadSignalements();toast('Signalement traité','ok');
}

/* ── PROPOSITIONS ── */
async function loadPropositions(){
 const rows=await api('/api/propositions');
 const isN=['chef_centre','chef_bureau'].includes(CU.role);
 document.getElementById('pg-propositions').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">💡</div>${isN?'Propositions reçues':'Mes propositions'} ${!isN?`<button class="btn bgd bsm" onclick="om('mo-prop')">+ Proposer</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Description</th><th>Type</th>${isN?'<th>Proposé par</th>':'<th>Destinataire</th>'}<th>Date</th><th>Statut</th>${isN?'<th>Actions</th>':''}</tr></thead><tbody>
  ${rows.map(r=>`<tr><td style="max-width:240px">${r.description}</td><td>${r.type_activite}</td>
   <td>${r.proposeur?(r.proposeur+' '+(r.proposeur_prenom||'')):r.destinataire||'—'}</td>
   <td>${r.date_prop||'—'}</td><td>${badge(r.statut)}</td>
   ${isN&&r.statut==='En attente'?`<td class="acts"><button class="btn bok bsm" onclick="repProp(${r.prop_id},'Acceptee')">✅</button><button class="btn brd" onclick="repProp(${r.prop_id},'Refusee')">✗</button></td>`:(isN?'<td>—</td>':'')}</tr>`).join('')||
   '<tr><td colspan="6"><div class="empty"><div class="eico">💡</div><p>Aucune proposition</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveProp(){
 const r=await post('/api/propositions',{type_activite:document.getElementById('pp-t').value,description:document.getElementById('pp-d').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-prop');loadPropositions();toast('✅ Proposition envoyée au N+1','ok');
}
async function repProp(id,statut){
 const c=prompt(statut==='Acceptee'?'Message d\'acceptation :':'Motif du refus :');
 await put('/api/propositions/'+id+'/repondre',{statut,commentaire:c});loadPropositions();loadNotifs();
 toast(`Proposition ${statut.toLowerCase()}`,'ok');
}

/* ── COMPTES-RENDUS ── */
function openCR(tid){document.getElementById('cr-t').value=tid||'';om('mo-cr');}
async function loadCR(){
 const rows=await api('/api/comptes-rendus');
 const isCC=CU.role==='chef_centre';
 document.getElementById('pg-cr').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">📄</div>Comptes-rendus ${!isCC?`<button class="btn bgd bsm" onclick="om('mo-cr')">+ Nouveau</button>`:''}</div>
  ${isCC?`<div style="background:var(--gdp);border-radius:9px;padding:9px 14px;margin-bottom:14px;font-size:.8rem;color:var(--br)">Le chef de centre peut consulter les comptes-rendus mais n'en soumet pas.</div>`:''}
  <div class="card"><table><thead><tr><th>Auteur</th><th>Tâche</th><th>Date</th><th>Statut</th><th>Extrait</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td>${r.nom?(r.nom+' '+(r.prenom||'')):'Moi'}</td><td>${r.tache_lib||'—'}</td><td>${r.dateRendue||'—'}</td><td>${badge(r.statut)}</td>
   <td style="max-width:280px;font-size:.78rem">${(r.contenu||'').substring(0,100)}${(r.contenu||'').length>100?'…':''}</td></tr>`).join('')||
   '<tr><td colspan="5"><div class="empty"><div class="eico">📄</div><p>Aucun compte-rendu</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveCR(){
 const r=await post('/api/comptes-rendus',{contenu:document.getElementById('cr-c').value,tache_id:document.getElementById('cr-t').value||null});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-cr');loadCR();loadNotifs();toast('✅ Compte-rendu soumis','ok');
}

/* ── AVIS ── */
async function loadAvis(){
 const rows=await api('/api/avis');
 document.getElementById('pg-avis').innerHTML=`
  <div class="ptitle"><div class="ptitle-ico">⭐</div>Avis sur l'application <button class="btn bgd bsm" onclick="om('mo-avis')">+ Donner un avis</button></div>
  <div class="card"><table><thead><tr><th>Agent</th><th>Bureau</th><th>Note</th><th>Commentaire</th><th>Date</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td>${r.nom?(r.nom+' '+(r.prenom||'')):'—'}</td><td>${r.bureau_nom||'—'}</td><td>${stars(r.note)}</td><td>${r.commentaire||'—'}</td><td>${r.dateEnvoie||'—'}</td></tr>`).join('')||
  '<tr><td colspan="5"><div class="empty"><div class="eico">⭐</div><p>Aucun avis</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveAvis(){
 await post('/api/avis',{commentaire:document.getElementById('av-c').value,note:document.getElementById('av-n').value});
 cm('mo-avis');loadAvis();toast('✅ Avis soumis. Merci !','ok');
}

/* ── PROFIL ── */
async function loadProfil(){
 const p=await api('/api/me');
 const el=document.getElementById('pg-profil');
 el.innerHTML=`<div class="ptitle"><div class="ptitle-ico">👤</div>Informations personnelles</div>
  <div class="g2">
   <div class="card"><div class="chead"><h3>Photo & Identité</h3></div><div class="cbody" style="text-align:center">
    <div class="photo-wrap" onclick="triggerPhotoUpload()">
     ${p.photo?`<img src="${p.photo}" style="width:100px;height:100px;border-radius:50%;object-fit:cover;border:3px solid var(--gd)">`:
     `<div style="width:100px;height:100px;border-radius:50%;background:var(--br);display:flex;align-items:center;justify-content:center;font-family:'Playfair Display',serif;font-size:2rem;font-weight:700;color:var(--gd);margin:0 auto">${((p.nom||'?')[0]+(p.prenom||'?')[0]).toUpperCase()}</div>`}
     <div class="photo-overlay">📷 Changer</div>
    </div>
    <input type="file" id="profil-photo-file" accept="image/*" style="display:none" onchange="uploadProfilPhoto(this)">
    <div style="font-family:'Playfair Display',serif;font-size:1.3rem;font-weight:700;color:var(--br);margin-top:10px">${p.nom} ${p.prenom||''}</div>
    <div style="margin-top:4px"><span class="rc rc-${p.role}">${RFR[p.role]||p.role}</span></div>
    <div style="font-size:.8rem;color:var(--mt);margin-top:6px">${p.fonction||'—'}</div>
    <div style="font-size:.78rem;color:var(--mt);margin-top:4px">📧 ${p.email}</div>
    <div style="font-size:.78rem;color:var(--mt)">🏢 ${p.bureau_nom||'Non affecté'}</div>
    <div style="font-size:.78rem;color:var(--mt)">👤 N+1 : ${p.sup_nom?p.sup_nom+' '+(p.sup_prenom||''):'—'}</div>
    <div style="font-size:.78rem;color:var(--mt)">📅 Depuis ${p.annee_integration||'—'}</div>
   </div></div>
   <div class="card"><div class="chead"><h3>Coordonnées</h3><button class="btn bgd bsm" onclick="saveProfil()">💾 Enregistrer</button></div><div class="cbody">
    <div class="fg"><label>Téléphone</label><input id="pf-tel" value="${p.telephone||''}"></div>
    <div class="fg"><label>Date de naissance</label><input type="date" id="pf-dn" value="${p.dateNaissance||''}"></div>
    <div class="fg"><label>Adresse</label><textarea id="pf-adr" rows="3">${p.adresse||''}</textarea></div>
   </div></div>
  </div>`;
}
function triggerPhotoUpload(){document.getElementById('profil-photo-file').click();}
async function uploadProfilPhoto(input){
 const file=input.files[0];if(!file)return;
 const reader=new FileReader();
 reader.onload=async e=>{
  await put('/api/me',{phone:(document.getElementById('pf-tel')||{}).value,photo:e.target.result});
  loadProfil();toast('Photo mise à jour','ok');
 };
 reader.readAsDataURL(file);
}
async function saveProfil(){
 const photoEl=document.getElementById('profil-photo-file');
 await put('/api/me',{telephone:document.getElementById('pf-tel').value,dateNaissance:document.getElementById('pf-dn').value,adresse:document.getElementById('pf-adr').value});
 toast('✅ Profil mis à jour','ok');
}

/* ── RESET ── */
async function doReset(){
 if(document.getElementById('reset-confirm').value!=='CONFIRMER'){toast('Tapez CONFIRMER pour valider','err');return;}
 const r=await post('/api/reset',{});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-reset');
 document.getElementById('ro-email').textContent=r.email;
 document.getElementById('ro-pwd').textContent=r.password;
 om('mo-reset-ok');
}

/* ── POPULATE SELECTS ── */
async function populateSels(){
 const [bur,pers,act,taches]=await Promise.all([api('/api/bureaux'),api('/api/personnel'),api('/api/activites'),api('/api/taches')]);
 const bO=bur.map(b=>`<option value="${b.bureau_id}">${b.nom}</option>`).join('');
 const pO=pers.filter(p=>p.actif!==0).map(p=>`<option value="${p.personnel_id}">${p.nom} ${p.prenom||''} (${RFR[p.role]||p.role})</option>`).join('');
 const aO=act.map(a=>`<option value="${a.activite_id}">${a.nom}</option>`).join('');
 const tO=taches.map(t=>`<option value="${t.tache_id}">${t.libelle}</option>`).join('');
 const subP=pers.filter(p=>p.superieur_id===CU.personnel_id&&p.actif!==0);
 const subO=subP.map(p=>`<option value="${p.personnel_id}">${p.nom} ${p.prenom||''} (${RFR[p.role]})</option>`).join('');
 ['ma-b','tp-b','mp-b'].forEach(id=>{const el=document.getElementById(id);if(el)el.innerHTML=bO;});
 ['mt-a'].forEach(id=>{const el=document.getElementById(id);if(el)el.innerHTML=aO;});
 const mpS=document.getElementById('mp-s');if(mpS)mpS.innerHTML='<option value="">-- Aucun --</option>'+pO;
 const aP=document.getElementById('aff-pid');if(aP)aP.innerHTML=subO||pO;
 const emP=document.getElementById('em-pid');if(emP)emP.innerHTML=pO;
 const sgT=document.getElementById('sg-t');if(sgT)sgT.innerHTML='<option value="">-- Aucune --</option>'+tO;
 const crT=document.getElementById('cr-t');if(crT)crT.innerHTML='<option value="">-- Général --</option>'+tO;
}

const LOADS={dashboard:loadDashboard,bureaux:loadBureaux,activites:loadActivites,taches:loadTaches,affecter:loadAffecter,personnel:loadPersonnel,performances:loadPerformances,'employe-mois':loadEmployeMois,telepaiement:loadTelepaiement,signalements:loadSignalements,propositions:loadPropositions,cr:loadCR,avis:loadAvis,profil:loadProfil};

document.getElementById('l-p').addEventListener('keydown',e=>{if(e.key==='Enter')doLogin();});
document.getElementById('l-e').addEventListener('keydown',e=>{if(e.key==='Enter')doLogin();});
</script>
</body>
</html>
"""


# Assemble full HTML
FULL_HTML = HTML_TEMPLATE + HTML_BODY + HTML_JS

@app.route('/')
def index():
    return render_template_string(FULL_HTML)

print('✅ Routes Flask enregistrées')


✅ Routes Flask enregistrées


## 🌐 Cellule 5 — Démarrage & URL publique

In [ ]:
!pip install pyngrok --quiet

from pyngrok import ngrok
import threading, time

NGROK_TOKEN = '3DBfx2OHicObJFw6nhmsfhx73gr_4mPuktvUFyfj4c4UjGaC4'

ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)

threading.Thread(target=lambda: app.run(port=6006, use_reloader=False, debug=False), daemon=True).start()
time.sleep(2)

try:
    url = ngrok.connect(6006)
    print("="*60)
    print("TERRAFISC TF est en ligne :")
    print(f"    👉  {url}")
    print("="*60)
except Exception as e:
    print(f"ngrok erreur: {e}")

print("Email      :  admin@terrafisc.sn")
print("Mot de passe :  TF@Admin2025")

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 6006 is in use by another program. Either identify and stop that program, or start the server with a different port.


TERRAFISC TF est en ligne :
    👉  NgrokTunnel: "https://chevron-remake-impotency.ngrok-free.dev" -> "http://localhost:6006"
Email      :  admin@terrafisc.sn
Mot de passe :  TF@Admin2025


In [ ]:
from pyngrok import ngrok
ngrok.kill()
print("✅ Tunnel fermé")

✅ Tunnel fermé
